# AP Model 3D Foto MAX: usa toda la GPU y la RAM (A100 / L4 / T4)

Foto → modelo 3D con textura HD, **esqueleto automático compatible con Mixamo**,
**pesos sin huecos**, **6 animaciones**, piezas por hueso y exportación a
**Blender (.blend) / FBX / GLB animado / OBJ / STL / video MP4**.

**Qué hace distinto esta versión**
* Se ajusta sola al equipo. Con una **A100 de 80 GB + 167 GB de RAM** usa el perfil **MÁXIMO**: forma octree 512, hasta 150 000 caras (antes ~25 000), textura de 4096 px y video de 1080p.
* **Usa la RAM a fondo**: carga en la RAM los pesos de los modelos (cerca de 30 GB) y deja **todos los modelos cargados a la vez** en la GPU (1 foto, varias vistas y textura). Trabaja en `/dev/shm`, un disco en RAM, guarda en caché los resultados y deja que Blender corra **en paralelo** con la siguiente generación.
* **Pesos sin manchas**: los calcula Blender (método de calor) sobre una copia cerrada de la malla y los pasa al modelo real. Cada hueso mueve su zona completa, no solo una banda.
* **Turbo**: forma con 5 pasos + FlashVDM, textura turbo y Blender en dos procesos a la vez (archivos y videos). Barra con **porcentaje y tiempo restante** y **monitor en vivo** de GPU/VRAM/RAM/CPU/temperatura que se pone **en rojo** si algo se satura o hay cola.
* **Malla reparada**: quita trozos sueltos, cierra agujeros y orienta las caras.
* **Cero vértices sin peso**: si alguno queda suelto, copia el peso de su vecino; el informe dice cuántos vértices mueve cada hueso.
* **LOD para juegos**: además del modelo completo, versiones al 50 % y 15 % con el mismo esqueleto y animaciones.
* **Visor web** (`visor.html` en el ZIP): botones para cada animación, cambio de calidad y modo AR en el celular.
* **Multivista**: si subes espalda o perfiles (o una hoja de vistas), usa Hunyuan3D-2mv y pinta también la espalda.

## Cómo usarlo
1. *Entorno de ejecución › Cambiar tipo de entorno de ejecución* → **GPU (A100 recomendado)** y **RAM alta**.
2. Ejecuta las celdas **1, 2 y 3** en orden. La primera vez tardan unos 6–10 min: la 2 compila Hunyuan3D-2 y la 3 descarga los modelos y los carga en la RAM.
3. Abre la **4. Estudio** dentro del notebook, o la **5. Página web** con contraseña para usarla desde el celular.
4. Para no perder tus modelos, usa la **6. Guardar en Drive**.

> **Aviso "No se ha podido completar el guardado automático"**: aparece cuando el notebook está abierto en dos pestañas o dispositivos a la vez. Ciérralo en los demás y usa *Archivo › Guardar una copia en Drive*. No afecta a lo que se está ejecutando.

## 1. Proyecto (código incluido: no necesita Drive ni subir nada)

In [ ]:
# @title 1. Proyecto (código incluido: no necesita Drive ni subir nada)
import base64, json, os, subprocess, sys, zlib
APP = "/content/ap3d_app"
os.makedirs(f"{APP}/ap3d", exist_ok=True)
_FILES = json.loads(r"""{"__init__.py": "eNoDAAAAAAE=", "config.py": "eNq9Vttu4zYQfddXEMpD5F1Bli9xXBcq4FyaBkiaIE2AAkEgMBJtayOJXJLyJi32u/reL+sMScXybpLdfakAWxQ5PDNnZjgc3/ePmGaZpqSk5OTyhjzh4Gp+DgNWFksG/0QwuShKUlFFaKk5+dgwktF7FnneIa9JU1MyH8QxyRmZxuTkAPYOJvs4gBnEUow0inagzud/np5fzDzNc65ICb+K5wzfIArwOa1Bg1xSXGZ1a5yxcs3+Cs0WwRRXHmwQkhnh2omiyiCj2YqhAYIuixpMf0/6OVv31arqWZYVLUsKUNTT7FE3khrOZF3kjBNFSwAzCiXoKZusAKrOBTTyfN/3ikpwqQkY4UZq1eii9NpPoezn1cXFNUlALmL1upC8jpZMB/78cnSU4pofEr+f8VqzWvepGOV+z7u4uU6PTq9e2wbLX+1KweYipwp2H5wd/350/Oput7yFcA98cybbN4B4Xs4WJF2KJujNPAKPlk92gI8jqbnMVpvJhZ2IsianUaFSuqZFSQG0xWgfAcZ1RMG4FMJTZCwVkkOa6IKpIO5t7ZEMwlQTEdW0YiG8Nde0TCtWcflE+mT47t0oNjvYY8aEJsfmBZHb6BZUKa8D5h9e3oAj4ih2hCWt0rxQD0FV1OlCMpYu75Opsx7CfkilYHBgILW0pPf0A8e0Mzm3STBVkBWFdFKCQuL8jN81DwngZjzC3PnKnagJXGJTKEIDUjgzSxb4Larfi4xQl6fzuZn/JSEdk7e9nQP0M5DNFme8vyUH6VLRB5YXUgV5CG4slE75Q3ItG/ZiLEAIFX6v03ObkoLqVfSBF3WA+Q9p2BpjdXzbio32TeByU8naPLMpsoZogkqbxHanmbBHM4KDoZvnDAp6NqG2PJyJRlmjYZRmvKlBBeGSDM3yJy4fQohalboAPqcPHCAXHWMERGc/nll37Nh6aYtln/wGH173XOQF8LClMvFtrYQU5ZkGFcneYAgKNRMq2YtDkq2a+kEl4xifcCtENq9oxlQy2DPLBCodiP40CcGDeMyTYTyehlDTypTX6CXj4xdg+GJRcponv9JSoV+xSKYSkeMp4CpaiRK+huPQuIRJlYysw+Ea2bhgNJ11HDBGB7zOfH523eE9RkO/4j38Bu/h/897+ALvYex4n43JcPwm6/Pjo9MO7dF03NIeb2hP3mQ93ZC2PF8jbVn9AOv9YYf0YPICacVmX4DtkOsxGey9Sfpg/sfpYYf1cG/Ssh5tWE/fIj1+hfQgxuD8AGmTCd/DeeCO+A7BNqhqshVt251lQ6VpYrgooGeAu6Lb5MB9AbcOVEXoOWruMNa8XDOJLQdjEqVA1l4Y0Jdgl8NkVcBVbZoQVy8N8oG9sR0MCCBuCYpI4JZM94Ua36N9IegEwyoBaPBuu6teZBDErY8lzLRPqTFY+Xe2rmEiT+JWCroudFf6iRXLld4SGsdt6XMTcAw7l8Gt7zxo9lT0MehOhcQVDhE1IqeaBRi0TTnH61hCGc4D/ArJoBeSrel2djvCWMcT/LPxgxotk+3ivUFoZxBm674Rz1eNymRxzwLhbhu3Hix87FP/Fre7YPTu3WcS4NiZjd8nBz3y7z8mTXClu4Dz27fxwkcZNBolMIAg4lpoXLFDXIN5e3LMvB26+S8hV1RB94Ji5tzs3s3Cz9hsQ1aBdNsK4zqMEUM8vgTT6X1QtvUpbIBL/D/sVZwi", "engine.py": "eNq1PGtz20aS3/kr5pByBYgpmJRsx8sLU2VbcuI62dY5SrK3igo1BAYkLLyMByXK5ftb+31/2XX3DIDBi5K9Plci4jHT09Pv7pmBYRhvkiLJFsyHHzO32MHPLOJhyNnRMTN/LeNdyeOj44ND9ogdRlt6X4ibosw4Xr4IReyJzJ5Mfs8T5gmWCbfM8iRfTBj7gZ0nXpKzEP6PEk/g78dSeDxmLs/WHN+JmMFYv5z9zjhebMUtM1OR+UHI3jz/++s379gjgMTY89Pzd9aU8TxgccI2fMdEDu04QigywcqY0xTYDsHkwboM4IWwCY1TGCgVgBXLBUsBRRw9VmO/f/6GmS53NwLxT/k6iAEqXJ4GcXljsVDsYI4JYYEz4Jm7CbYS9ZRnPIR5LZgCmgEiaq6s4JkHqIh1Cd1ziclJyIqMr/gHeC2iNIHubMsR1CNPbB/lm4iZXpC7CT4CzCw1AUkiwj4pxFowN4lxKJfHHtDxPytGMPMltHtIXaFNBpQRMaFeoUo9dRKxtYiBkG6QxA03GrplIi/DgngFw69LnBQ1RIpLsklUGcwGGJNvkHZhxY2GbB/KvBA5UioVRZBhkyjII9VQ5CyI84LH8J9I7IlhGJMACVQgwiBTGeBq+yUIHjQFBrl+9X7tVlc4ehisqtsgqa4+5ElcXSd5dZWJ6irflEUQ1nflCsjsirxuWWwywb0gXtcPgqjufBukIK5iUt3HZZTuEMU4nfhZErGz16dMvXsd8bWYyp93aT6RDWzWTNUP1pPJryfvT9gSULVTXmxsL8hiHgmzuuerHH9Nx8GBHceyJsAz5/Tdy/+CXjWy9mniXpnW5I/XJ3/+Bi9MA0aLC2PKjBV3r/A3FD7dZ8F6UxjWZDJxQ57n7CQGNRAL4p4nfOY4QRwUjmPmIvSnKIX+8m0Sw1TCZL1MsyAuLNka/2EjO4URsR0DsZDzsj1RCLcAjFoNAQI0hb/tx5mIVvgCh2m/yTc8FeNvnGi752Wxt+PI65TDBMffOFdB7MFrkNnWyw/JKofHnwzuFmgyjAWbAbFF7LhJyPHuc93hO5AptAhgKoUXoLqZeAFKkmyD0CL9BS0BwQMj44JRBqXxOWhme0hRcBrRT7IIR3hm44jKYsP90RN6sJLmAh48ndkaFgRDajVA+dzlCPd28PwVD/MOGRQ8J02SEFq4vn1OUngG9yc34BPAx5gRv3Guk+xKZPnyEMyLJOCFoZ4Zl41ggKRH/Armn+WmEp53v587x6/fT5m4CfLCSa6W51kpQGQl9Q7+jX/SHdXCDh4iTLjnXAvUilzJPNitJF+aQMvYBVv0SPONBpG49zjaGpamFWDSToVgRe0Ta0+CfofcEzHZEx94Fta+EUyrbWiSRRZjU65BQdc+d4WzKVeV/cjBHzjEPgdI13QBK4GCeHFZPyqyXYMZtQE1zcAGd2CYlk0TbzfGf4HPMnrnBB72G2lWwxZbagVdtkEOzmakbYPvQ0A4LzLTt8nKocWzCJQvh9vS87yZk7hxRVqwE/pBd4bRQXuUyuCYxtuE3GnpJeDhBbmjOgj49dUCWCqsVtdMgArFbDapn6KsZJ5JiLWHiYHYs9aT66DYsCQVMTVHi7uCIVal7wswnuvlzEJs/T5NrjcwSYaiPkyvFQwlVdN8+pj99BM7nFmDDYFfcVKw1TjZ1QzjwQYx8gO03FxZDQXQJqIrtPGPZtdptsNGYP6UZipu2ngUScHRcuRlZIobUP7UzLyplASr7zBM3ziTGqMCLxVLgrIs2CcJDALWH344mi3suf+Z/fICXxv9qfnGJ20G7IAVC3vmf86NZlBFF4I6aVyi9BlpkArNQkwxepgy6LBKNKGQSrs78iDYks4GLiqtrU3GcXD+Kkyu3/ACDEO8PgPQYaB5HBwLaHRXextHc4AyEGrCrWd25Fhh6SchmOwl4Vvmwsm5D/45zhMwz9K0aqJDE2qz7Dv2CmKFzR/HQHJPuIkX+IFLUbxMIP53PruBSwgheRqoAHrU/FTTs0XMwZk4PoLeepEZuQ4P18nSiFyjLdhK3c93qTjJMshhegAHRxkdqa829zAofcPCarJgpgLBfJrEAQw0YFLuCf+LYSuBxVk28oouTQnqNUQPEOwQl9UdxTH6A2K4/uBjycOg2FHY1/ZqL1EB9SzPZufYm5lPwKOhlj6s8bYw0hchtkwoYwDUICbURItSBwheWA75CiptnUGxl0AizIPWGaQ94El5tIIURuaUVUoJwGtQKhGLEVwI2MwkOlUwpbB4CfOCJKrlZKswPwEX3Vg7vLMxdIZgJ7fd0uNgqIqoDG0Q+OTaKfyjQ9BPJNqeTnH8pc1XEFtsIO676rZumRUZMCvMXwCEdZYA4d4LiCBFpiuzzk8W5BTWtsWu1WJZR2qAt5PEzjotjUuM69GhNNJyt1+AsbXYfnDkVuzfm0UbVjM0A4FoR/njwEmTSGZBqphe4lBSK8WPrBa3bNvu2J1Bq9LJMOhW9xDDYeOoKx77Z2xUd+/AC4qD7eHB7IAIAAagbbO/0r5IEtxluAZEpCWXXRlrc+ereCO5MgU1J20d4EsrN7wnA4YICk8pw+mLGiSWOJNxsaME8itmB4CV7G15FgCXIEQueP4VwqdS2HtNH/KTL5fAQSGMtt9WChuC7JPEDlvMlsiBdRphjNXn2lfyrM2tfbJ5N9v+f5k2xLK+nP8b3GrxaYqpVZmjk80TcMBzqvGNcE/WVmqWyNtBfrS8XSFuhkLoM+zeC5wJ45KKMeaFRgsaTLeilx23IoA+kDZbEMEM9zMue1l0jvl4TOPtoZs/JFSEPTM/fU+jf4/I1JcEUSL0vYxovv88IGl7497RF71K1zBB92cWPaEbNrmt1GM4V707tNekyJCsUxRa7EUJ0M7FN6XA4CStyX7wVcEQ8B1O6iFtvvpW6ch9nPn+SqceutX6ibFfX0drQE4uQImo7gipZXArk2QMIuGhXurrDXBhJL6P2YpxubgD0SqHo0DfcdPSUV07yRxqzR1xj1rTev28TS9mumDBMqxk7yBPwQSjWd6wjJY5Q4oMRcr9QluKwmq27P+06670B9Zg+S3V2UAWtH6ih6MjKbBdJKYBJDOs4SpvK6DTzNYbtZIXgtOTRZc7CyhVbycSmGZrJZQhGYGJw6U25Z7o9cVOZrx7qpt912F7ARbfQHUDkqGMKtgisyORb9Rd5V7ewKP39GQyIoiqSK1PBvUbfu7oIQeqOyjxkU87zkXrrNBbapiZQFFehgWgDk63RAOx7IL7oshBn8uSWNLHBrnXnefXlmFdTOd5hnk5ZPY8Jk1TKxfKbGnFDClMCEiTlB7fFf/SnNb3Bt60M3z8t41w4Yo62NsgKyCexKGSbNcxKpigynRdVgKojYOpvcsL8AxWVYGk+lnTMMgdvuVBiFIHjcijzyYjQQIurEIk5XH2aRvZZS68Vl3zET7tlDtnstz5r38OlTtlyROLKJ/WCoISku+3GY+c9er7S+yuaW+Xn20KpzzPv8kaTIBrorHI9WWYVBmEIFpjeRVLAM5qLauTLfsADfYZhrY7w/a5XG12hUmw8yKzupKD/KWFWpvq9vBAs/bN23dpboubwHcgIojBbeSi3XTDgd9husF1OXhho69iyyUz3v/y4rlBZjtObZ5lfEcdLyCmm7KjSzsKYhCPn9jhk5mOek0G2TUpmhGGZtDUUxA6Wp2tyAoTRzes7oS6LZ5rYvAd7nEAlQENBWlLVh8E7bbA2CdDrZX7EUruZegkcZdBBAGuaIwyEmBwpnWLHeRQN7lsFifxrcgSk7Of2XzW8rO4CnGTd/h1M4O+8ynbwe9uDjBucknBKV3xG7za1c926tkAxYgIWZKaJsLcEVyI/+cEF341ql1P2Ub1aJlAnAOYaVzxNLGNxX5gc3v+THOJH2vpisW1IvYUYgKQRUDPhFHVf/rqx0cw/3mhZNY0c/C212BmQPWxK9xt5J3VqwVDV/AL6GLN+ezw8ZThX0vtRLBPn799+Y93v1nkaongh4+lYco/NkY3TyGzhvxUiKJy1Hg9hVwbo3m5u2CKWwqmWMCcqk0FrVrxcbANPLldZ5N8wMVOlT3L7TSxwJW/XGCBFpd0sELsJ5Aktauzvhx5r8prq9fZesXbqjCmuW0zLMeQgZ1mMdSs0WZIImlWgt5Z1qietZQA8aq1gMhet3NpHZ3bPN6ZMz2SWoOEACZARMSAZwWt66pUfqo5EowzgymjVVcRl5HIwCthNNs1dD7b1qZEQh6INjsjynWBoBNkK1JJcPtB9bYR6DO0eQq09EDAcTgYx2rp/xDkoW5oJ3C+Wu8/iagUGF/ML1tkRTrmMoGnVBIfoaBBO1Cs/GKGHPoTFHlmzx5fdu0RNidjvSBTCcLCNiXYy3yBiw0Fbm9aQySh3Ju+Toy9SYFs0i/TmOryomNnokkJAIM/wdnH1pRMDDxAm1Q/lUvkAS2R83gtzNhqkMVdRAQqsUF6g9SUrRNs3cWh6ZWURXs3CPZBUGB0pNm1EMAtwKMBpoRxR8wAyAW+vayjXHTvqAGasVU2nJ6isQLaW1avTKUUHCB+k8CDCstNTgJo8SBTBg5zAfjLbxzcapHfd0XXPBZyV10hXkE/tXiBE8Ebr3Th5p6h+CtIZwuRKRANERAzVKLWa9My8Xmv1SA6vcaj0fMO20G82noZoSDVr2xMRH4TRcejRqCUnudgI7Pdlsa20UYGLooM3Uoi90DguyBeOyr08UoQUoyxnaq7uacLctOBaMKBlCLAwpMjgGd7uriQ3Qpnk4CyogOnC0yBDmez2Z5xnCTD/ZRSUBw32VBuGXazBswuIlzYox2NkjLWYGaSBfiy3VmyU72yz+WvGREdxQ1MEV5RpBMRLbUHakfjsqNLX5SssYbfwwXfCrOBKtNg6XHPVKvpAoUhihzjtIQsqSFZbfvBjSPro3lHvkc7hGHF7l77Qfoc7q8lZRS+0IbaJAqKwJPp62EviNVknv3cWJnFEM8106EUt7FL4N2XjY3q2kkibRPEUVEIrZcycdtAXKPJLkQKP4lbZIL4KbwpW5eAPIQ8akNLL/EayqGjMiwC9BdXC4gE0FVcqSiERrKDQkTARhl3aAWsxr1UizrSNxI8pM+8CY4gvpC1/D01M3hwWe9cUYX8/RW3S4v91Q2NKlzUQkAHBdqippXadJB7Bm+NXFx29/fE4qYw06ZQKMfqFvwaPuPmHSIX9b+7sqjJYU6rhUP9in3dfOrZMRMoQGjcIMHCwITuwfA8s2TMVj1hPy0h0JU0efJ15dOk2Iisg3Cbph3k+3MZLKgquJ2iauspkmq4oErNBoqq9X4jelV6vLs02FSGRJQWO7n30uym5lKtAEnFbaIfadSF2tbdyFFTmfpFuv0Eq4NbsJ9LhQEkvzEWtlDNJbfgwups4JMggpiqpK6ssqMrkc95WSQuZKNqUlPmFbtULOVLLMIX86fWoCmjtU0q9ywpkQXz5TSjkJQsNQkaDZSkrdLrndhLPrUkWHdTxld5XQeVt/vqoJXFc3IXovUlzcOsHgLQdUXQ5XoPYmWRloVD9DCUtzEsyCCGTPNUcfSbxLKqXFpbe1XDbUWzyuBjvPC1hXYFqZ3BBX4Nk/3HEihV8KKoAmkDK8UgJJ3dY3uWi2r8dEWg5KUt9Ze4NisfXV1Kf0OmYeyMAzke8kSAoXll7dkT/R37Qy7wZ0JKJ16bfkbHZh7iCSQegtratgUJnzy+lOHWZhYmjHvg+kU3BaqTH07lIaJirQl5FRfgNXk8qeV4C8Lz1UFbdV4La3KtHQsYxBGeIN5TuVQvJzcQv92JfAvDJkWTO0K+jXAnahcpSvaV2LUDmCQt9PwMDY06D4R2f64Z1FpG6GBMm27oRnXp6JdK9i59buwy9bDOcmWDuJC9tMYbbcEnrHYFRrUXiwX7248aBetGeHbJ9soozU2cIvIpKxyYvtqlOzCQov7G3ogbL4Bkp8AB5oeXTRiYlbEin6zaMdSW6kgPKIy6JJ1R11Taktf9MpmztwbYLd5P+5GDGkRab3VD8ay6BiGu8ABXtZwfHj1uQ6mM9JJOuMC4S2NT4jJS4AlAYEOHOZZz+xm8TMA0ieUMGraZD02TpfGSQwzDM2MqH6CDqYbmURo2KAGRC1zFGZiTMmVqA20mwkBsccSneihNWdk6w7Qs5NHK4wyN9KJrJPXYiFZnB1W9teh9VuvqpBuf1Z4Vt0idXRh03Tp7QyxQLdWNbCpv9LbEIdVUXsuWdK03lOuE2AyvZKPORoCa2Kphcy+b1/d6J8WRal7qTs1M3rXQmHV2pjaxt+KECVJxCN7iv8ugoN05VH42dDTBOsgzVuSAFlo5S+lSLexWddROb0R6obfpxxHSZ+mdSKPagKVq6Y3oid7q82DJfNh86ZmajJb1cr8qeTeqblWJ3GAkrZvQ4UCiRU703GB3J10ItZMfCUZwNw/46LCU24RM4xWeiKtPn3pC+TONfWhDYSwvcJUaLFuZ77JKgKXtob/S+MD/yvbU8fJwBFgboiZ7RmsE/9dWSP7Uhkj+TEfogzaJ/ur2qL5qjJL6HQbT2KpVkoRmfWtNa1MlX8gbq7FZMgRWd9YodDBpFeRVovmiq8oSocPWXbXO7Kta6OQZLEq71GHXIPdUOVa+BFm5MOARqPVYyPO+OrOMi0rUCQOaq8HzXC3AjTnAM4naCdwPCWS1Z/KookODt+DpxxWxSe+QYqtyPlQOWXxhoLG1c75tTgQTfnJk36DgzvkUf7bTeI2LCXXvk4oZQjs3+h17jwUr0pcQUvMZeyDPXGAY+7EUdJAdt2egf+K5qM7pc1CsLEgy7ewChognF+rwKQbm5slFffj0UmUIKF9q1wPuq4QW1WlU7PG4oZVTbebGFRd4NbP/NsMz9M0Aj3BMrYP0MlXHusedWEg4rYN9ReBemRAVr1DpXSxQ57RPIEk7vDIM4/mWx7dkdla4sMfyEngTUZbg4fI4WSL6QoEkX7bmrfVMIt585MxEcwiQggBAAHeQ5Fhq74vFjSrAwMSfTJnZ2Zg1x5nimrSc0XwgNq3dIEfuraAXx7WlG5p/vzmhc82DAkZ8YrUpmEL4Lfok9OOu3gKM1rH1ky0QrUOBYtNqIg8WmgXuMCiWyKspg+t8OcKyKfM4eMV42d+eDtpD64XmPerVymb4caexD/FiGA5t8Ub65L1lETUuaa1GtfrQ+EV9RvwSa4xz3VZWJ/3B1LlXuM2osx9Hi2OOIEo4iXEtmS+qz2yAzmJ0yxIUS3lAyujUfaoRBsxrF7kDHblOq+rQe2cKo8SNt+r8Z8v0kcnU7OUWvX0+lFaRXx/ToKr8RNUWT5VLSUCBTqAqymZMNety3xU6QxbZMFasz64dHRuoP0b3QIdBwfy2ye2Nkb3C/X8E7l//lOcijE5xGcDcF12ZaSz0iiitBuxZB/iaQyIDAVF1LHaAdbWp71isfNxA1RwzToMoDYj+uB0pk6zg6/KDyJLcGJaD+riFWu6VstFZQGmtQ4kbXOwY8rgGz9wgDLm9DlfG1whm45FGtu3Llx6w5Ur5OCm7uqebzafSA5IM1+7uC5lnnAVxUcmyXjh6/fxLz520BK0qRHZLkAPkGt6sPUKHoUJkI1WyzZ0yVcHdx2NpKlWJMriF4HKU3WOHLEa3nN57KeB+WzJbWzPv56aGrHbLtmtBF8ZjT9px2EP1SOnxpC/YMqQnrrWHb4VmGujmYQMcnk20sLX65pFfCvpsRlh9DomZytntuCrYt794RHv3bQ3SMX1JABfI8bMcKsBdQMgG5pVEgtcf65ji/mRynnJTNYiCTMtyu7NhpiVF3Y+YkEzmReaTXBoP/udB9MBzHvz64M2D3xwD4+KR9IJKqiPZRbEaO3cbp12E7hBrrWzE13LPkCFJgaV0mrAMpZtUEqsukORi7d+IIUgD6hlqg1zd91KHW9BHZnJngdZL7rxyKBGkIZvaBQY6Q981akWb+MmmtUldp1W4uuh/ggNhDYVpgMwFdUYhpBBZewQA+zrIt2sVr1DTJjCBGJt2hOFTazLiu9AygUeHOP1HsEl0j3E2AFWRdpPVlIXcm9b9zo6dl6sIIu9+/KVa4QaxdCqL8Fr1oSo7qHKCVlWoawmdWmbF4boogGHCVBl4IjY+WObFomaD5IHajZYXDVsbCVjJDz8Zf8V/xYYUSt+x5ffG1MY0n+QBKWDplqjJFzWDoSeR8mE3/dG/XCK/+GW7Sbprm3v89BZ+oykvSi9I7E0RhRBbIRHbWbwvF5l0179HnQCnogTK9g/MddN1WSKoVye6uVLmDiuy31upqBqRrchN6DmQL+pkgBadeQaxn7R3/gU9O0LShbVIXMJNbFytaJ+K6uARdPNnNQgtc1AVW+62T/U9n+l2/7AOifB9Bt92R98Ojb3tuHWaGqRx6ogNsZ28AfD802erWqoJtlTy7Ly9CxTwn5peXFqAZ4E7WCmNAEcEUlDD1Jp9cTwOlOBqf69EoAeys9iX27KITLRoyUM1V9qPQKK7VBlVVUnkTsCX0tVP6493OBQqLOlbDmZRhe7zkayl7qUg1v0I5p29lCmoenWswH2GxRM1w91n1L1f8RyGqL56uJQrd/LDlsuzC0NeYRVxnZb4gHa6NJSuF/1MpPmUKbkEkb82aOMv1pWWhxAJxDmumPPcDYKl2gOuRGuJG9S1ujfuM+6rkKIVFk4RMsK3r7MAOIz22RpaH/nbM6wrRCn/WAqZK/zj9Zmmd7dkJCAGesgM+zZIjXZ5QX0v0f5HkL6CX/NWTmvavHh95hyfvDp9fn5yPIXkI0px4BD8Tric09ejbvtLF2SPcWlUeDhNPJ2IdWO0ZwN2D4ECkvqIv52/e39yjKri2/jtFcTVNA07Sh+jZSfjjL/K0ksnC/F/Z8+YnsUMzqjX+lbRu88cHxjiNxSQG0rwzur6sSzCdF35gmAdJyATApdEup9yqtk4p+8AnuIhztanrurFEaDekrCASSyRRyiJSymOV2K3vLIGvhooF3IybZt75+xnp5KGLdXWgF+Dgj6FF+LhPiyYVuH9wc8QFmH9AMMlGwQA3lExWkXs1P/N69MT4ODbE/zQ5afmxImKOxfswjTos1TAOyr4GAr8gs6Z8bp+IxsNq7MJYQHIfCgK/NKFPT/UwZCqExSYQtNuDBIgf7Orv/xH4J7p4FweumVI8GSLUUCUtGTVkRmEdNSeHzWQx3VHoUQ8lV+9VREegnk808FUL0ZJIxMlr6oqOjwOIrhFSE8QoRPZAGf0y+kLVr2+Nzxw9ra/wk1E9tMOvFcv/n4POGkgbjkR6Efk3Bndys/WQuR+H0xqXv2IvHoDNGs4eO95hImHIJ49BRB/yA1DAvmzps8Zm6fvjq17A1NWCHwSqfNzGXho2mNYWgVI5W0DyvAHvmipgshdEY8owqDQHWpwavGXYjcGRZ1iJknA7dIkuziP91VqjZgQ0jilpt1+eOsAsvOjpzNiVA9c/XYEiMS4oihNqNEeRczPzUYa5T+rg6pfm3BpSdbAqoO2e4SyqCVukAUck9ipd5B0No24EX5g4kLVHV6cnrw9PnkPEzog73Vw4HO3SLLdAa19lCk9pNx+Rpdn/WrfUJpU5aOom+mOAoiDAzlAgOwkesANkMSQhOlChZfQ2VAEgztJM4PO15nyRoKVRFQv5I01BE4KeU3s6gntHJGd63sJuNonIl+qu0HQNZNwivOqCK/KH7IQL8l30JhRwnZs9RxHR4YairFNdkzHa8h5Nq6NwnZV4ui8aiosjY/F75V0PvuaZvKrI+o0iH1GYSHICo7vAX+W+svXZyf0HCIJ/flv58fvfj+/qxqMobva5bQqfTpBM29w83BpZ9k5rogIYxiXZrbEprcqbjynI37Mz3gk6JMv1Ef1lQt5MnIzXuFXAwZCv4gCFEjBMEszsxZEZv7lPbSAGQhtsE4fyUJmo3fyy0XIEzrUrSzs8A7xppeJi7S0RmsfQZhM+ylt/NJdas6pgvR4NrMGDeHAegboeBHEZbsATsyvTkQScTJ15K9XKeiR7gI10urOtD8pdcTUhHCwrrm1j5tGubUY+xptwH5eSjnAgQBGxc7xT/goqaEjj+O7rzUyd9etQbJotVpTkmRd3BSyDqWqUES6Vhkhxc8Wo6bjTksscM5Gtp5IaBcHR7PZbNHfDytfTyYT5+TtL6/fnlQf2plM0JWAdjuCPrxudr+qvg6TFQ+Z6jZRaFVQepujGvAnDbw6nVPIVMD+D9eUk0Y=", "blender_rig.py": "eNrFfWtz28aS6Hf+CixcewPYJEzqZYcOXUe2ldh7ZEsrKVknKhUKJEESFgggAEiJ8np/++3HvPCgJCfn7k2dYxHATE9PT093z0x3j23b55M8ykprGlpv4jCZhrnl7Hk71vHFuZVay6CwklW4Tl3rz1VoFaEVfgknqzKwiiix1mFSBkkw7HQs+G8sqvfGVq+0+lbvVL7y82juZRur14P/Qb1lOg3j1JvHULKXrkprEuRZCEAvPc+76nTeB5NwCDAHnvVhmaU5fIkDwCWGf8vwtlzl0V0wDbr4dpWE9DdLkxA7kUWAY2KFsVWsoJEuoWZNANMcqlgbi2AFMQCBSuE0gpfOwHvZt5YAIwcQs3BSpq4HFXc8611YwiPWKawsT6HEJIKWCigXW5NVmGep5UwC6CSgs0iX4zwtuoBqgn8Al8JlBJbRNAJapIQqwIlEf7qA0SQPsR9WWACN47BMrUmaWAkCo4asj9FtsEwZkLOkB6Do8H2UQSNAMtfKAugNjVCUr4OE0A2SaBkobAUQQY9fk6jcdOEPNB0DCmLosdO7nnUaFmkxxNGeBPFkFQPAIgVkAMkAcEPk85DQJ4pOwpxo66zT2zB2EXtuBUi0hO5Ar6cEisir2GyDDcCoJMUsCnMYM2OUES3POiwiBpSk1gzfJzREwXz1JQQ6Y9PY7mIF+FpLYFPo/8q6SwnLZRYjSzlQt0hjQT3swDhIkHegRAp0K4IxsENSwkh51nlkBTAQQRFxezCkwQaQKbIghl5A4wxmGhXA+BOgA+AMiBAGBVJvz7Pe4nCaxHfyMEuLFJoMllES5Nh2nofwtwji1TTgHyX+HQdRHOTEfPuedXTLzP/L8RsGOAUgP7/53LXeHB99egeMB3zCEwLGO1xmAbBACSQp3K518uY/xGCfXxwTL94hDwP9CNuudXzyrrDiaB4Sy66jAj7dhGOg6jqahtydA886C4m51mFeRhPoDU78DAmOhGH8JisgIJBBleGx0IPjdWzb7kQ0l60gnwPEIpTPXwrgF/F7GZQL+Tst5K9io36W0TLsyIdxtpE/k9USBAxKq6wzy9MlQVqVUVxYosTHoMyj2671n6ugDPMEhqZr/YZTPe90LvrWiEB7+I/jdjodkAMwtHPnaeAOiYxZHiWlM7MvYepZX43CVs+66A8PvMHsW3Fld62nwDazeFUsRhf5KkRYT0DM/c3/kGpFB/5ZA6ZADw9/XsofXgQz6taxez3btZ5Zg+GVFc0sfLRgtGQpmI8w4y6vOkEGQOQweIf5fLUE+XiKTzl0Psi8YDr1A/EewUYJdGwaFuXIjpIMfufhn6soD6eijy1VQK4/qhyQk2DPglUM4BcrFJ7AgLbgXuO/ySJF9hpd2sGqTKGWWdqygQ2n+QpkOn1KomS+SlL7qrXVRRjNF4hgucnC0SxOg1IjAeqgHdW0BN7ZUqnv9ZsYL8I4G9nzKE8tmKYwjUDMzUFYwmQJYiAL4Eoy8Q+7tUGaiCZ13rIEuae0D9JKYggMa3Sq/7LfWq0IUFS2V9prJwQIHND8oN3bW2qtE6dp1t7ITjuxwzgCKbKV2gfbqD0DrXwXkIJeBhn+sAQsVAY5KBgX5C0IJ1JLyqQIXlkoBVC8yabbyVUG89AcE1AU8GjwJj3bIQlv/MWjeLUNXy44hAk5WURrUmtr1C8gccJ1BGhR/SFpMSmaLbsBjP6zz7FHyR2aAACJ+Ay7uA7vkPXA8pigpgLNi0IdP5Nwj3LU1dMURpTIFWQRTKLW7s+ComwdxX5X9CeDmT4OwH4gXFHXlQQLBBiKncwjsYNACwfFktuBDi2Da7DFcnpTeCA7ujBhgAp+ei1Ex4dPP59A/a82CowhyUMPfgJ9A9RdBby7vPrW6RRg64VQEJSDB/oRB9ejd511XHu9jsIbPw42Yc61vJytoFmGmO72/7boJoqEeUHaBMQnWHeT0gcGcYRSmcH0T0lGU/vpGKzrshiqwU09UacIS+fnAOS31E3BpIzWII0c0ORpuYBGRo4roFZbUg3dYkNcVrdwa7bAlG40rF+vY4mjRwggoVOB0Wyyytdh4aczB74JVEDvvwVTLCgj5C20V6T5h4sMLAiK2BNVXWD9PW/v2fN97xatarDOXA8tB4RU5huNdR6iCALrpSgRhgJABcLbSQgrmsMSFP54VYZHeZ7mui5ALbiPsPIYoT6UBCJWQCLNwYqC2gga+Iteo+i6vHI1GKxQQAsZVqAiHj0atJXFJmMTKJVCdbUIkiSMx8G8AVsNP2D4bMTdnIyrvRQkgCKC/mh9FX6SOTBCAtaSxmf8xZsGZUBvJim8STIPjMVy48BQOGDHSLvNtZ5auzCjaXJDIZK5B3uugKXKedCpMJgsfOiSY09QQ01SV8OfpDCTikWQhU5v0LV2+dNHbjmAxcIGcYQ5jxaZf5Pm8bTSJwDyD+vj5RBwGe5eeRdg0vDT7tXfn5JsDgZ5ByUBqCTvZgnYBlN/FqAxuEGOL8F2KJwVSCmik5BA/0rJAoVwaAoQhuXCK7I4wnos/cDCci8HV16c3pA9BqYcFgcWcmxcNKNOgb/lzBbDLHvCXfO5RSzgzKI4xBZGCnAHdBvDG40ADIyDXQUC5ICXPsN6GMBsfGvfgwV8bodRhFwrDyIwSs83RRkuj24j4KcZ6mhYBNPCjcZqCtrPBiaARkH8LYG1QiTjZbpFfqLxm3rIxYTjx6Pz9/YVkjFJYYlB1bc1DmpQ6GFsv4xwxGlJWoAKU62ZMLQgditkYFxQ2YHm9CdxGOQOTSz77fHR4Zn/z6OjU//i7PDT+c8nZx9RQUpADP2yf9UVDV2CQU9swPMVX7nWazDz2xr8kkYJMA08AYka8hrfe0lAgsH+SPswtu4XCZoKKQWHIUGtqCACwhctqbBxFC6qmTxcpmvSStPUXyVxlEgFrgkF8qlTw5r3AWDk/SDLYpBM6SRA1UB1YRmBlrd+LCZBHMrl1RPrE1l10V2QD8UWUFfu+KS8C4PGzx9oolQ2foC+EbQLhmBg9X5HArNVQWY+91EKT42gwwtJ70yg5OBK04OmIijgGABgDb4HMxVMe7dDYrEioDtLWH8ub1laglHv9N0u/Qxu4WdnAR/w5/L2cucKVpjLBP52rUHYOwBBxMu3wuNVjPXcWnQmsLSdbOCDA0X7VzBboGr/yoWPO116ORAvB/yys71r50hepwD8XRTE/PICS8XcZaeHzfUmG/iHMHNdV8NbZVMc5sYYg0aYhn6xTMEGgWWz4GdVLUvjzTxNgLV/svb6+B8vWD+lIGFbCPge3pCGcibp5bBr7VwR8Vw2FSUWUmeNKm0pjYebMnlQ+6owqZrszDv+cpSnq2TqvEfN5nZwk8AmEQFyGVu+tCV0sPkt/SC/UoP0iX+5mvGiOQksWt0y/z0B8hdgXuTz1HKAt2coTUILN7jigDdlAa3U6r3mHaKYttpWMGS4uVUGbOlw/Sq5BoJcwFz6BYoOFmMaocrCmkaNoIH88fr7YDG852Ey1uIEgDuLxvqVZFd4YHSQZHq3E7sLkyiapEMgkixJM/sj7tVEQQz2fhHOgfq53Ideh7SYfrsByVoAPx/Bgi2Ev7SPFRYk04o4JbUpjA2C5ONLIbqXuN6DR/VRyrolEHNqLT20ABIQkkXV7EtIB3j4xS/zMPRqZQSURKugN+fvfvZPzz58evvh9Pjond009SrmrflfghpzVRaX9lm6mi+SsADm8cSqy18H8QplOayH9xvVhSF8RH9g3rY3QPbwXzKrgNfKcIK7nMx1uBsuljq4a+5LjgCO61rvhSoBwQFi444EH7IdqjnJgPLXzpVg/iMQ4iEI8DwJhla2Sko+kiCWx5U67xaHaIDD1AiSL4EQ/LE1Dr6kSJb3BIotdWrTQSN0XDi3KGj6Xn+HmNi1/o/l3NGbgx3jxWt4MeAXjNQEpPtkoSYTAZbih6aPFGz0xeXJ0ff2XiIMyWCoRx14SYCtn0YS6k84kvuMkB4t1aQEIz88sZarAhf6X5A0sOTWRxQWET8ACwIUE0FaRJlPZGdoz7Dve4SUIPYpKEpeC4VhybRigvQPsBh3P55h16HAJf1DA4hE4q9529ef5NfZrR8rwsEoEG6JE8+4oCZdPFN06/+o6AbV85bqeaN6Lqv3KvXLNPQ3CoBEcFAfN/yg6/PYMwHDMN4CgPROCwBof9ck8WEyWchtJ2brynkUf+DjJLbtUA4x1yIriqHrqTFRXMtfBIvG4dy/JXbpv9w32Q6xQ5AGb41p6QLvLukfNZ5ef6AH3awe19apsrH6sIxjOSwVnJDWSC567mI7zO34c0Ds6EpiXaR5AZZ/QDTjM70shN88DcEILg3S9L0X/do8fvFSvahO+Z0DYz4Xiyr6WI6AC/Tl4D6Fii/7kgxUQg3yYKCoLMDJbuIj9fKl7iW3Lkvf0UC9HOwbfAr9NhhN01ThNTC4vYpKv6PshgQlAhJNyc04/AJvrYCPZmLcAkVpSfKp/97awN8f99+LPaSZtQCOcArQ6KYsqtB8r07zH7UwxYrwcAsFiSjPNNc2uGqCA7PTr3KW2A3gQxpHwnPqwLqSYEjcfUYIXtJumW4H0MbzG9qAmNPIMLiJYHkESGMwYWEOpsR7wbo0Jqw9LiONegIrOnoHMGGZE8RzD7eXnQnMzxJ3eILbqBgNhJbZr84mhIYAYGQDtP07WzrNHJChJd+15MPAfCDju2vhYoiHfhF3rfQapzWN34BhL3J8m8u3PfFa6iKsgLIGihiDzTac2F+98mBpFkJde5wHd7h1HM7FSYbQ/XRkvCqCHNTPlg1qvVNdOU0P8Tx1SgcbBhGm0aR03o/A0GbxNuI/XVZlI/q3S1plhP/Qzxx/5s29dhL+I/q3KyT5iP90WTaNhERC9hrRtG3AwNmKn+4Uy40U6yFVAQ8kPv3MR4sc92rfnHw6OqdtRg3nCR5E4Ok+rj7G4R2dRuNpcwZ2ttrhldaTP0YSOVJov8fNdwA4vbTf4yoC/gokbGavUzTZtZcAD8TijqsQzUTBApnwcgG9ga/PlK5Vj4OXlcedF5qFk3ByLSXXgaEgg6l8+6OWZ2+wGUVMx0a/BTD0nT72hDrA42rzLGTF4+oCheB4XAgaSzLHPs+iJLSbBY3nHXzmFptVB3azrPG8S3W5kWblHbtZWD4zeVTtQbX6J/hqtxSWL5iMqvZOtfZ7+Gq3FJYvQAZLXcMtNWpfpJl/lJhAjDryFSzuJBRqUUDRm+UoPwH5YBkyx9P+pDOA8sfhDI+IkNtoUthEGtwJts/Eqav8lsM3Q94UaOTW5D0UxdmIrC64BWegfWUI50kcrJsVpe1Vq2fymAZxk0dkThRoGDvUnx48CLV/oMqF8Ti9UeW4li64Dy8lFn1S+6zW4afR1hvc07+sSBeHtuSewYgv0lUM5h9uqUOvUBy18kGl0mG+tLkkocejUoG2rebPsFbm2qIm9UgDwG/b6r4PiIdEDSKZJh29ZaLsanCyOQOkVomzWxZRKMptcmCggQQzg20b/pTbusaClHKTX0iSS4YB6UX28u5+bciD5DoOjfqzW+ZLUgr2FdfaE8MnzVRd/ToJsbaDSDxjYLSvVmWB3hYeGMPCtdk2qSfJogeq6UG9bSi3tXK3jb3vY7lfs+MQXTAWaLNgr1oEZqUCF+eS1G89wAxrO7elOPlFHSSBrnlfvYs0fBMUKOm5DvTU5CkA2mAoYUFcOqdQRnn6OAtX/SxRKOHXjCzpTG82urzJAxSBpkApo2R7cyW08p9yL0yoZXNrAw871VZZZW95iAp+tUxQ06/COE6rmh/PPXm7Th13VvZIGjsjG2CtzUCtEDZiCdvVz7xoYWtjg44NYufQ4XaZNY7hPYDpQQlebqbTjVz4k3lvWK3ju7ZFHlRgtPR6BN+p5ciBNgNuCxNAFua4UxPFoVp0ye0GktpXYvnVtV70GbZjfHW9INk4ev0F7461rSE2ahYBDBw8Ug+FOXPctcQjyQP9uC8f+2zsUGFh7ACn0XmTtFyU4aHNCCmir4SvWIgKiXaupb6MurxvGCarJSzvy5DYuzC0X2DM6X6X8YcFB3D9nSk5Wkuh1xfJTPoFhOKVIXwTRJL94l5WIL6Rhr0DPA8cOe5SB4wSoj+JsvKYSTbQjoBLIkrQa6E5BcvKzTFc6GguwS9q/IxdCo2MtJOMzjY70fZ5wJ8X9FnpTrcOXhhSD9SvfFZA2bBqgLwIorgGkrmr/3IbygN8yfwpV65C+ro1U6tIaiZWzapya84JMxAhswQXFTlXlNQTnG7/DOu00jbNC8sR+AoK228CGgKJUXV9XqYt6ve26FLLolMNtSmcobdXlDO8WmWclvdV6e+01NEDM7O/Fsm3r7PkG+suwBzP/qZEnEfUOl+QAyTVAEyg1QpAt7JqffP3fRQCPAFf5WEHfkg5Yh4S/dtI+zeqA9rKEZI+jNFj9g7tq9adeS0KxGq1tvZ859a2BMzFO53vfL0eWg6fi63xXIxEUREl5Ccd4jsSCGLG0znzMsicOFiOp4H159Diun9i3a61dmuD0vYf8vg1lEXmfudFZbgsHPcbewAp9wKzU+2qmzdNAjz4E04cdAYoh6DwkvDGsY/kaZUtj8eWZml5+l0r22W4Yq+PztVB3aNfFXo9yUp4Ru4AQLfqT6De1I5RMXqC3LLwx8g+evfhQiAVom74+k3JDTRhgGPLWAkBIsWwoksIQw/UeclkoT4khr7xUFQD33slSDfkDQJp7uA54ltPFHY9kPDzcgHqYRD29qpSQ8HhsrUVE1uvprrz8pRM5b5eg40vE+S58eM7igZekFcRITDCPwOgwSP8vmopgqeAkzSBdTqW0/uK94/LyZv/OHorR4aJPI2KLA42Ph8Lgty9+PD2n7bkJ69YpDd+lPgzlMvwnbb09JzjIAOabahAqYduy3mqcYbaNUqStzRBUBvsf1k+of9/obaJi2tAerzxRVRE6JTQflga2gKM2jMRRDHkylaxCtYhxyOY0RS0MV2Ec3Q3pYMSjh+QVrHc31KeAKIpvSmLNlWXQJIl2JXWoGQVcsVDatOIEMc3D25B2Yxp2KF7Kcx0rEdeMx4ohuImKheOTVsoLT576AIWJauw6jpg+r4x4/vo4hJXxdykWgwnSluxAOctbjMHVX3MtSdxlDkOLG3gO/pvBGNcl+KCAOrhI7mR/OjSbBtUIf+JAgGmJB0yoNq5Ar0ajCtliMRSSTJZqkCQ+PJ7bW8ckfpT7YzramK8oDSwwuTamdKegoHcjXBbvM2cHn2FLjnGqt56+tTaMYpfUpnXjAsujsBOug7DbBotC3YcgnoD7yUvtcl6uKoImRvr+ci68YrVsqXqM6TgYKfCWF8etO/JgRYdKphpyQklvPXnoPpAiqAvJRjrAKn1M0llADhKqsSOprdMmiRN7sI8dW5w6L7IMzz3sn/V8EeNEE+o2GTe9Rwdvp1LPCqJ3Cu5nry5jBAmypCzo9Pjw7dHWoi8pXgwGQNG7uQiaiwnH3Se7uj5SxFgYn2U3m6Ea+okzTbCSZlee0ITK2+cRgHpvnaKT/YjdCtVqzehnD4Kj53zaurXqES6nmuCbI8wXE2o+rMQ3fFsogu5GcoKpA0Qyd9OPh8d2/ItRcj5RXSH394DDw/28ciu6d+8hHEIMkAlos1+WWi7R4fyam5qJUJY+NXJxxG0oGeuWJMZxNHOWK9hAQ7/6ZamW6nxLpwgKd4dvf3w8fDiyDbm99LL0XUMahI06Pm2BiuuhY/oyNTsCGlC5i50BCR1193alOGC1TC9QHFQlStXLnoxFsHUypUBa/f6RGuAfT4Pz4Aiv54d+Ye/XpzYbutgYviaYbbqJqtHme2HcjObJxoqVA61pAhGMCWG1tfwm2hyFsEMwcUXyTUSBnMUBkydisRRnCHw0Ig9QfULxJ8Ib6wkxTDJlhBBrZXXwmgxRkG54Gm7Dn2tK7itNW6VOlJJrz0TV4Gv7CPMGpSmDPW1FPPJuir1tlH5Pkp/JZjfWmIilZh79eABKNlFdF6aViJMtTlkV45oG8OgBo9WU20jWO1pSwnprTvXBGwYc4YcFGs/ZYbazGiOCOlwbYNdhFOg7owxPym8WPlDCnja7YPcUYHD8npEcOdRRhzgUDHe8EBC+C23qFxhwFRJ1Syq1K9p8FQ9mknOlUJz1STjqZBE9rvDi0Ph9X10ZstKQmxIuSrfYicQDV+oRCV64Bu+ooVEQUUKCo367Zezk19P/f86+vDL+4tz+5ssTEBgvZ1FCTl0np4c//7h08XR2an/6ejw7Oj8wpZFOdjFX1O/fRkElKNlah8eH99fbkpWnP3p8OOR/T3KCOBpqm5zKSfSNN3KBddoH3BrAzNiGYEVBfyzJ2KzaWaJ+Wo5FP7+nKPfn/+STtNy24Dev8ZjQvunh0BMMZj3aQWTpaQfdIV+DBXJDJYXhaOM+t4+RtFmMKtGO+5fNAIqDRNx/DItg3h761RotOc+CC0RdA8p2Gw7vHRy7XMIwkjEsX3nEroq7hoz1K0IgKpsoIk+ZxbDKAZc2Dw04bVkFAzGcfC5Fvqo+uD/U3J3URrAKeTbKEjgVxRiZPxGAAmLMsr5k5Rs7pBTKiSGtYyxq5JZUaUQ5KVw5n6CxvYkSFJKkIAOXgMwqP5dOA+ytCyiOXmG65wSHhOxEhrvXU/RfVlGyP/z3QU8bQkcm0c7wu7+Oudw86HFJNUj06AiC6AbilIgs+NShKpr7W6ElTUUJg5ksnFCjyGgy3nY26O6IdZV6l+c00grwwxp4+UB6DSi4QgL9aiQxEop2b71ky74ExZEzSGLaX65xo4wrZwKILfTWGQ1azMEoEIBGJqYXkZXHu5IRsYR89QbBzFpYWNpW5BzLs6SahfYvrvlo6rqpked0o0dDzksiDJAfuSuht/FVa8P7QGmswj3fj0ZCFjBSG2pi3bAuHYEQ12GPIiw1pTj7D7CcrI0F5g0/HIlWOKqPhpd6BiWJnyaIZoJfOfet0Rh1tkaNwV5mQxY31QWxdpMkjj5kqf8XAZdk/GES2z5yVVr6THUogAlmafDcmT6DZzgq4Rjw+/Q/R2fuEwi058ksGoUgZi0hfiVp+gQmPuhWXo/nzTmXIOFjFmKxz4tLJSUl/VBv0K/hEEb1YAAPnWOiAV1Of4Yc6oQgydy2CaIFXyXJwCIy4Stf2n5JY1tuys56Rnggx6PwohAKY/pZTgckeIwKdyOobhuzca11IijtSsH24gJeiWtE4GGWKsqcCoETe1Sg2JjS2XZbmgeimMLxE2uOeUe9FKbmQjm759S6Rw3ncNPHz6e8+kD+0lkY8cwrB9zPCW3dQzXRfTdgLfm6Rpq9iwtQkOzs+nIzd7kKb/oWk/hZ6E9MzBeT4SqgKjCtE/oUv5FpHJartBVo/eaYqc4PYxFO6/0tboJjbuvsnuyd+OmXSEQVk4AH+nMA9GWkc/UgFem/p+qUSHoz6Dsf9ZfIsPjzil60NBMzlMzV0CtjjhVwQou1cC94DOTlB9BGJMgmDr0DWP9BBnHqyie+hydL8g5y3lrHYnv47lyUYYZmKQqCrZyekY19bJJmde0ppkF16EPP3JzTYMDSxyFNWnF42NirFLqvuZ30Qqy86Rsrg01lwwrp0kctunLDbr//PUQFkKfPpx84tG9Dje8QwKcpek+U5YnrE7nIZ5VSYIgHVzcE77kN1e6OaSJMwPbjL/gPjz/0sttjpLGl2T4ztz204q2ztQ6pJmojXl0BRlaq8/TDUcPaAa5m8fYKWvYMCesYVaguYsoCaFbQ4onyLh5NGDOFNycIVBtBvlWm6N2rhzEZkanluOYbQRaB3H1qK9yZNxKLbPKIwcHmInHVth7dgsytmCl0axJKyn4t9KopQmJ8Ta4xOBkv5h7GcTkpEtb83i0hijOJnX18/b3t8dH58bG7/0hiGq9WgFNCT2oSzTXcA6p6fboFTAppEvsH9oPnMnDBCHVyaSsu9ujkEGXBwZ6ToHYmKMhYvenLJJvskjFWyxQlL3TDs9d9SD9VYuF6e/5Tjs196lsxaf5CYrrVZwidiuhgijmgjYKEQTvjC1AS4I5Agt51Fzj4AttefyP1XuxI7JMCW+IcE2Bd4Q1CM5kx1nE3h05LsPfLh0Owq9b8eaWTgh3BTZUPa9Vz+vVe/Cqvf40vUlE87d4ZNeVCD2zKqHzL3bMCnmjQr61wueu9XvX+gPHecA+BMKRfaB/6rPO9wseK8wJoc6wYRoXDpiIydyPMU8A/8zxJyY7GaGHW5dzeHEB/gkF3IbG/1rhcfLVQqfnIRsn6hnw+r0rqdOTsBHdz7L92K25xLKrlwlNvWBwPUdQTwLM3QrEvAER0ZF+2SaK2jUca/eQCu3Y1CtXXm6p/U1TPg4xDxRG7wCyXesaf13jr1maUla1RxGYHY9N/KVbM7YfbKNktZrxStZrp1e9LV3jeltLjXbMOttGJS1rQ0L+00RRDDZ4is3BrEBKbRsbA4R+U4ORt8AwRmgRZYW/ypzpnWseAqFPoBVgcCFIogDdN579McQEnGBUC9v5dwtju2KYmsrG3hboBlNqesc+8qrhEhWrU7YwgIFdNI2rZdD6OHfwrAdE9VPLsGE47SFMdBEaMmb/fPETY32NspfSuRf1BxNQBw0x9ehAaexWKpFXqa4io3X+UDEEJmroSODW6qOnJdaXVIdqHIWwoMbqpMg0JW6C+LpCCTwz0I1VyECzrici0M+dgCJT9UNzB6bvvUDsSSR3sRDi7h24VLH1E7rWR5XeyZwcYgx291XbvcoT+qg+TBWOzF6Qg0rhMFkD12W3U3KmXTSHsz6aemxeiubdZqWdeq0dUa3HXru1as2RyVfJ9w7MC5Mc+qE5MAPyl60Rfw/Hpf3Lg8NyYI6KfgBoD4/J7j1DsvsdQ8Lza/DSlcPzKDLfBOvwsXTmzgqPw8GOVuhCf9ZsjYMaQ1aUnO5AXff9LqRsTwZTKJo8UmggFfZdBam/7z5WRP3elfFLD9Lty2qZVej2xOrjjLSCeTBZYLZGcstGrrSCKKenF72BNQkojXMeTlZZmAe5eQJTcrDGsOHfJo1pB1242Nkbhue56b1FmZpxkFT+BWSpSXUVA58H5KdefU8x2dJdhs1Xhc2LKjYrNBhLDgZxCZm9JgpCMBEGCvUVIe02EAIIGKGOPvT1sm4TywqDDfb67n1NNNenBvovGP3d7cQmnFZuC96K0PfRub+Fzi2i65qC/g6AXdWva/qfsue2SR4xGXe0nY0pseW8rDjqt4mfVfY9soWiIK7RA8vbuXdyTNml9fuk905F8LosS42XPX7bpmb3G1Ur73p1HVVXqoaQYb1aeQG97W45XlGLG+9H8orcU6JfrXXgS8/48rA+UJNXYsCqYOc7VMHvXZmBR5BDKIT9diTaJenvhq5uCODqmFe2Pu0zSt0OIF4AW6KpKfY9RU7LamGZnblr7UJpNMdaS1Hydyi0A8MJlkFbmXNODA+F9l52Sa9tKVVyIYCEQryt0BtKKS8gETO3lTrFHb8LRKrLZrfo58BtxD3y/sijt0c+t67MeTleM11hwbXMRihxQWCg58MjJt2CHF9odUThTEMEzXqYgpeGMuKJK9eZX2hsURW+89JLVM0iHTHF9b9VpvvXb53aISKlUcwW7fui2WUiQ10ViyY6opZWnMuM+Rpep/UoGFGfopGqAESAEsm267rRl5LhB5ZDdapQsJqeKiJ2jWfX7vdMrs9dGa784ORqXbBtGVq51mPMhm1o7mgh0JRqHKQn68mkC5+NhdiWetSvYdsCbqDGBpdt3/6VAuPgkQJDREyVQzVnQP1TugZcOXOU2FKm6tR7xt8bVXR6cn6k/Q8fPLtrbJ62hGE1JMj3egCJs1l96sixMHgoQ7u9xvGrUQgPVfVJLReUQg0aImQaR5VL85ACKUn1GvsQL3aUhyYeXiacbrBWVp5rysGmqxn0gyjP/pIJJgEG2ZErRMXgYXDByNzUfvSZ2BMj4/beM0ykGM0T9M0PrDxIdNaxgBL5NXfjI8o17lHaRDpR39qoz/kW244qHqgz0k3IeIVHbPGLQUj+/km2SPjPidWnU5+eZ5jBj+LYjPB+cVWPvFiAvd9CzLKfBXkZ5dUrBSwnXuE4U3ATpuikNG9d6YqNSTrP0zHembMBvsUDL7psKMCbC1TiyzxfzYMCEwAUfNEL+raXQRyLY0RKhpmQn9gakwyEiWedh1YoLo+BT78cv3n+85vPz+nWGHWMPU0pm3VfHf8spXdINTyiwgqNidGaJLP19I78F4ysmXrDrqBkWMD5TqLzayYlp9V8IJOmzAtUcf8tTCzHhcqgyU6i9pUXFT66kNYdsxo4x7j8MQBgGgrrbRqnOQBBCEWnFjcGNTCT/DXwsYfedkQaRh+jdi+OPvsfPh7+cmS3FYMJMg/bZ1ADNfbJbYWgfcW6FtpHUNTDmBMT1xtQZQtyk3mgnezWzBjP1Z5iZuNqmvhdY/2O7WXRLUi0Sqb47NatQs1uVZ74BTlS7ekCizm5Q9/ilWCUBt76hw6NuwTV/+OPtDJ6+YIU82Dv6j6MnlC8Id4alsIabR1ynBKmwltmsQhlyFeRuhLkw2ENEQf/PEMMMBwVn8TpUv1lj97WjYpmzUFrTTArYEmqM7nOkUxOSzFYRzUhiiREuvamtXa/rXa/URtZ/8DYSEmYD6YcoXfZA9ye4rq+B83QD/iIChdm1nWIcN0rIy4Tw+lqwYCJCATcqcfXdcz7D0YSrqOY5L6RhiqaZaCuystkdi1azk1vFZo0fILNwURoUgvPZpvaXGDm8GwRsO+yCIvzx6vZLMxNf2YBHCPR0rzIgkmorhNQYWswt3ssQipVahMGbSDoi5fD2i923Cr4DEfASAdEQyPkJR/En2Ou7fwTvLgIbz9g/2yzvJA0I4Rmvja8HZy6aJHf8Hh367cNsNbujjEBkuU9uLE8/hhktllBSdtzoX9b8h1L1WxWM5E3+uJhtsjdPuY1M15udIsrur6hIUkFEnxA1SbuoV5VdkI3qRB1E74qgEU6uQ5LQqAO1ui4WRtKwuhzUaFvuiZpxLsttaGgqi11XrdNERoReqgtlTOmTMCIvmsdnXpCXKP0WiqNiqHI5jlaRoUvb1Ci1N/aoDKh6KZFBnWoZ9zVpOODahAfFTu3LXJLAhchcX/bcBRGVt45+fVCZjnHG1nwyovCdIfkgs51lExlVhZlV0ry0s1SZOTwjVH15UaT5DPTu4vavMQWroyrRcZgsaDkcajBKs0F9nRfWnv5v0Xsr4jLNzP8ULV8dHZ2ciaplxACTBpxXx5uRyEzgMyYBEmRsr6W1MYjZcwYiinjQV1GywzMCFy8bFv9qnWe3OeiNZj6ff8SzMzeoR31+noNGfuMmC8Clpzt1zChdd16sdI2zFVuiLZK225kkk7C1fV3691N4y/QjczgF1olAzfDipmvafXFzZN084zstA8PspttBGBPSgzapxyu6gaYrCsG0edbXkY2rE1g8HENwV1Ul3x06rvTVI0Cx6QWFu82q0zcCyJewOprkqb5tLgPEqv3olpTLXpG9tHn05OzC/vh4bm+kVvwEke50K0Bx9DKuoXT8KmT+yGyjr7L5+nT6xsFSy+mRZTV24sPJ5/O7e6jYicsYxTQRCErOJnXcKtJHkMSXID1VbtVqzX8rL0TyERCHtrAR5LBgA2QrUA0QoHMn41v72VLvFMYLx9SLIlXDT3Ekq2IYcWKFGVWRYCtnEkaLQ6DGSfMEdxopHa8DqlPoxrLdPUn9H5GsSD2+OqMWi2XxIHPV4tJxtffxQCCUCxB6PrX4UaOY9fYMCkXgk3enpz+jsk+l+Nw6vMmRSi5VGY8IvOHo9Y//YrBe1oQgbWOLd4E+XRk9/6w2YD3Yf7ZCBZzIYi4Rp8B/IxRMQYmOYx0viG6+WT7U70ihFXntPbhs23yCY501+IhoZER14PQ9Q+sCgJMOW7coixuReZ7jytqIvP5dqTtzCW1C92PpTgMHv6KbN8meFtu3RLdNZgQ3moBQtBh5AS4VvHWMsu1VOORbjDEPSBI4PrKzVctgJgLxFh9Ovrl8OLDb0f+H/fCWmVq0M2xRRp3Le4pdZjmf1HG9w6RGs7dKVhOsR4lePhXj1JrUC8MGTTVMmQF5qH6q0N2D83ncTrGC2zo8qtBv6/O0h+48LAm9HCWIuYNlB9Wws3/WnHSQ4sEWqInDLdADdHMPaXrqId0ITddTW1EvE1T2jPFZKUy9u055icDww8L4UYrenFv08mZn921cw7fgS0NGXke5fNr87COAVST0/D7+7LTiBIqPQ2BtWuhHltz1NxVgGi/ejM5jfioosFUbJGK2RUY6i0glby0HqwLJStRgFfGUnDJuy6YT6hoXBJp7L5ESWXv5ZGhpm0BhISrcPmWJXBbaDMSB163Q+t2a3woYCwji3HtM2cABmW2JPshC4dSIuB2GC5ElmqnAd47u40g0lqKp3lbjqdKZkDdMO19EK/5FNjGtz469tB2L3uDq2YmMMBAIAWI0D7Wi31xHl1tzwNemXGWOszHMbKcp5NGai9jo94Mu2puwidT3oSftt5yRfvc03v242tQ27fO69s5rQiboyYWlcvqgGAmH8FvMqlPLd0WsT6xx5WRUQbKKgatpseiT+quMA6K5lhdujMENMYaZwAy5hp4KV0lpZlSXCWPursvj9GW9I1mYRbSWnmxnTiyz4+O9YFoSwUUmNA+W1+YhOnsw+Gx/VhkKsetbDpyapEHrtqk1HhkhIooWyH73GoYOjtHYKmGVEi3Hjrpq5CFYK0XvUyraZ1w1602groBWUhkHr7yVHypbqeaCKIuL1psiof6l37f8p0gmHnbti+nDBWe3bWvrR+xIGxb49TW1o0EsDzAMhklofw4YnzHjaEdbcM6ssWu1NddUvSuvu6IT2Q5uRDnkWRzAg9Gi5VFshHXKHeTmK6twQsFxaVShUiP8cR6BwsRCrNnUQo8MlvhGwSS4nktxgtTGZ1rI6Sb0wNPWh/FFusDMWsaH/i2YnvcbLE9bh60PW607UH5hh5retxUYNxjetzca3rcVE2P7ZbHzTbL45G6GDpSVPUjSecBR8+zSnbpds4tWnMtEm3OVjCHWwycXZdDAr7HtCnJjwFzl6lMCM1EJXWxh5Uoq8lBSy7GiWnTIGDst0qdAF27pyFY7wJsrZhgaYDWVcgnU34glwr1/FQ/H58cXvhvT45PzvDx9ERnNqIBmo8D80SuSbr7T+ewPuan5MO59aSCIDO1eQBmE7aACNZrHoW1dYelqLKE8Mvj7MB7rTaRN/cRttS/3o6aBFW/ifajtN9oRrF9ZdblRF3+FsFggtXHR1Crefi0xZRzO4+01+qz/rGKFep9n/a8MXZmmxvQPq4rHrsHvV0L/tV9W0ECYk/etf3tqBkuvWV/9ZFt8OFzDeGtWdVuHtK7QjSwpiKtW9AS/vjk3RDX5gVf4Ub3C0bzED2UnGVULFO9FWdtzGQhvDEHEL6swjmCvAnHUGISxpg0Flib93HidOpQ0tDKYVmcNtVjnN6nGmNlOc5swNn/itY8ZyN9iqmz3G+PySEbp7UUcerFEhMgXS61CxU0qHQo+UzpWa5SslxVva4AyNA4ITN0sBgmMZEo92rl+z2JV42kq/RXvhQrxTjICpz6EQWbQ5dMUfZXkq/ee4a13GnF/J58NaJaNWnNFkERp/+Lx2ptYgX++V852tp2QNVw2dzW679yxrT1HOs7T5u2HVrdF+X00FESn9kkwAo1VwQY21LYj5gTY+pzqt+u9fWbe9l+2i1XM3F6X07iNikat61eOn4yExCbV8vTBeUY5u6XwZzvZ6FUj4jqwN/v020snHUAX+34g311AQuML4J+CvWt1yPMIW048flZ+775TG6cf8UWv4mFSHWEHPykfMzzEeKXAXsDzKEQyPTCRXXAG7m/4X3rKMGHVjDOQ75/3VuUy9hy6FZqTFUBqy1sD91cQ9QR5G8sFYIQ91RRLoSo/gg9bn/6t2k6IfmJ717/RF9iEFkjOyxseA6D6eufliGIfw5VhIm3Kme9l/brDr+mtK32OgpvsIs2eTjCGmZk30RT6Ns0XIPp2qOHbpREJcyuntjPRhhlVMbha+7m4am1++6n5/yq81MxySPQ1rzZAtRdxaGNzo8je1GWWTF8/nwyTbwvBZA9Wucg8MrnSbZ8/o95ms7j8DmNRw8RC/N/7Hr7Xv85psutvMfc9QABOvqcW8Nmyw00j9dsfV0G+TxKhv1XM+hUbxYsI5gGxaYow2VvFb0aB5PrOd2sMnwyOBi8HExfkXEwfBKG4beO2dBXIsAQ9OH65tWClhXDlzvrRQXGzs7O/s7kWwfmTf5V3GYxnMXh7at5kA1fZrev8KF3k8MT/vMqC6bTKJkD2Oz2Wwcs9DJNvlYgzg5mocJqNpu9Gqc5mLTQJf7RwwDJVUHAJTT4bQ32sDXsNXq2Dgf7ugEvnc0qjezt7X0D+hHZfnrOHIPkQw4xSGBFUxjGtRjCFu8HMKdx37eHHJTjGjRYlSnSwMLY1wJs8fSmFyF3FVG5GdkDm+ZVAfoNHjx8TNZRniZ4u0WPXPBGdhKu8L56HGETGcBtGq3xEsgCZP2YQgUAPQqdwLLw8XVrCZilqoBk0dcdQLkoreV6BLMJXcRLzJx2FIf4883mw9T5Ybn+we0ebv9OLUOR4+1FsOkf3FeiMXwa+T6YXOe+/6ozWyUcmjAuEyfrlt2Z+5ULjjVEzvglgDo/8HgCyLGHh9Vvxcwt4TlNJnE0uR7NXmVisfF2EcVTZ+y+Es5rY2DwNeY5OlpDpWOYWaA6csQymP7QddzR66+HsKSBd+8vPh6PfvjhlYPl1xgvN47DQ6X9/vu/L69cXJcewbrUSUavO9SHw24C4h+GfxI6P/g/dH+wgDoEFsHI2p/ozoZX8AoZxXG/ufC/Vx0kjgLpXM665RXURLDHQBoJBRlxBjVeddT0/+k5cS6wMYpD6cuP4NAKvWxj2y6Gm8TRNJiC7FtmYJAH9pU2QK+7VskKSOkejIJDUz619vv/TppI6SD49HG1YUs/BdXz75VrwUAzXXOWzbi+P0I9FsvCS+Grhld0ihUk7htbaYa6srl1ppUKImPf2C66oM0WhifcwrvJIzDLsZAaFluyH6D9pQDJMF0ts8JBVFzXPKekBnzQYrjQoYeu9QAepP0O88kiWpPXuIyywZ3CMsWb1ZdZAOsf9K/rZP4YP993oO1RCX2MTY9SH7YeQCMRyRXYMIS/Izc0nl8H69APCrAqowShmetkar9LHANrMmkC4rpK2Td/13kSjbnAem6RuyP1OicasmmnMzfSy+gueIVJT2fs8xFD2ekKfWjJaRR99zFPNknfMqBMnVAGGBXXmBgE8/b0VxX6Ur1rQ4gLQRZGwTMxeYwTJLk1ciVCMUUHVrxKtfdaoZUSWg28DTNMrEE3E5geIod9kmK+u3nlgoL/b/gzHbchzKYUxbpBoe9BUIXjrTJffJCuXNSAqBMmYO1wA5xoTqwH15U2K9QB5p0VYmuPLL/b0sN3IYCc4H7hdIpnRjZ3wb4yP1Z2f1lG2ienFx8+k0j99d1h/f6nxq6QwsDDebQqcUCRRvIKsHJLadCsomThuNvyYuvdQ94BnvJtGVhf1G3Jhyi70yw83LqYmyIr4ppXN9halgei/RusDYLrasbFe1MDNmKRHiHZtjDjL8AYSDNE798Eo4jrKRTPVKqSPyRt7bJfNj/KPPBBBuLqbogZlzEKIbXoph5Q9muSQOO0hKpfVgXlfbdg1Udp3fOCDBD8Bv/CQmmC4bBpC+LkGsiX/2jXTHNvplJalSwXGEuVxlPOf7LbLLoMbv0xWMQT6tpeC8mEG4EutNMsNIfVXLG5v0yJV3gsowK3Bh9RUtyWsx0/Mf0RN03Jxq0Y9wrP9mt7nlgnH959spxVQQkVgVNQxOmA2Z1n7ivrBAj8WeodkJvxfJWA8MTCU9oHNy5Iq7Egtk9pbkFuHH2iwL93R59OPpwf2fdVwUx92IF5tjJRf/QUUCK2SOMVbbjcSmYmXQsy9v7im0cUB1th6RvjV9FQXBKXMjoAip6oAt3Lgtuh88/sis8/EuOK8siDlmH6416hzqffBnYc+3kSeHglKR5DgV1pV1sCoY4eOz7lZLgSM/9nwD6a2EZgOXeLjrRBUqhzZZW07sY8ILpJ83gq9jCpiG2GCdBXzJPAFdtPi8bIljeNcx2MlKR7d3n1KuCO5/L0pd/iPYOJnvY5sQHnYthX3jS64qBZcSDC+55Yx6sJZcK2di3M6CfkEp1fUc7fOJ2AOYBZn/HuMl4tvI3BfMTcCDu4cdXb7Vq7Huaq+hFjrXbw54Pbg5gvIYbZhs7fDkLo7UBVPIHd79Mx7CMg4KowD2zOK7KLqODx7QEhYa5Njs3xi3GDQ+enpt3no0PDsDn2cLVIEZQhBhE6dFub95JvGjSKicvcKvG16ba7YbmtYyNcsSUdM7xxzRaNwip9cLiKSa441ctT33O4I6Vc06BdzDEONIKVAqYddth1G71w8QOBctxHOguouwrOV7Bo6TRckNDFOyKFBEswdOaeTh2kzQhRowQWAqmRTPgkL7rGDb/Gtrv4pBwbqFWetfcd2HIxBrzlsPZfe1D7aEc3mJ0UuDyQeecGDQBn6Wq+SGDN1VIf70qXNKl1vHrk+sR6GyzxmI2PgZf+1CQXb2TJpM1UUHSEiuL9vUie/b58t/WyY1G5yxUfc9MxFKwUI1TwjodgacYe4iMvC4JoiZUoiT1f7dq1ZCpSeN8ygxyV94zy3j8VlXr0epIW1dd3rur6d00ws+3HTTGddYT1Kp+yO8bxJjZpatmW5R7H7mr9twynUaDuFP7tw7ujE1taN9KS2fduH51Vow4f9wTEmRY28PPPH0+PfhE3Wnqz2TIL557+jB/3al8nMI/oArH3OwfNb5TLvfTRb9HnS6+o6Idf3tcboT8+bkfQFTj2Lycn71QZsXNB/nPlAkhNJC6MSxvfnfhEnXNl2FPAJS3rwLizuzLw0sWip2dHv304+q/tZXnvCAqr4eF5Ul3Fdu5Zd5sYatNpWzil2C6hEFmErYJYxeW5JGC9A/SL14AxYYvut0jfYp4M0jsjY02F1xLuDsdOanvQOJ+VU7OSy3AXs3zx3OL5crDjtt94gC6jlTyC5lfeMHMSkc9qIO61jtMUHZx6mAVBJN2g97MAyEDG3W4l9GnLaRjPwK+6VwD4Bkfsm7fM9swtjtbp2rpxos605B3cVK/b6NUzzLhgixWg6V7FTKM2j2s+HLxrKjDXbT0Q7vsXo3epFd0jxW0Pshd0GwV4iZvow+8/nNaxuiKVFi0Q1O/HxOryTWfC/fDHgy1copN9ZtF6q347hWV9iVZuNcvMAx6V0dr0wVpucZtUs6e/fbpUagNcE+kWbSVtqkqx7bdQUD11UUSt2jboOzJPqZlK9HvaMe9h2Hr3BAJsTRnlNm/v4jOHiUYgg4letuxoXWPISphnsGYXFoN9/AGvwLT/snwSjNYTmRPulTpi6s6jPN096P91OYMALIDwF0UH1vx/KTQAvhIX5CFQBrjVuI4C6/TTL60SRGjbvyUv/nJs/5b4fkXaNpusdbv3AduMEz99790ojzHKgKzGfrtpDLWfYpWBzwPiZcm8yUV0lAbcH8XxNj7SIPgq4AbQv8dGmmUUJ8mzOj4n61ryhEocVUEJAY7P5Dg4gfAVH8jhX16/R5srThktQw//cdCguOCspCC+ZqnymM2jOb9gIwGPEe2tKTvE5pKsw6U7951t6sbMk00mkTqydLADID27FvqDJyWmTIIVGgZrB8UkiqTXmFgtjooydzskMDA7Ylo9/SSSAOf/X3KcTls=", "app.py": "eNq9fV1vG0m22Dt/RaWNAZt2s0VSkm1Rpm40tsc2MN5RZO9sEI5AFLuLZFvN7p7+oCQTvG/3AgESLJCXC1wkyL1AgLzeIA/J8+4/mT+Q/IScc6qqv9ikpN2ZeLFUf1SfOnW+T9WpGsMwPgSpiGf8K7sRU2a+i7nrhR3mCp+JJM3gZgg3QRqH9CwIUzENw2sWMidchizicy/grSib+p7D4VmA/09jnoiAM87gaiUS+JRlAUuzAEDA9Ws/zNyZz2NhG4bR8pZRGKds7odTff0lCQN9HSb6KuaBGy7zO6GvEuHEIs2bJdk0ikNHJPmTdBELGFgwzx94S9HKO6ZBM57AVas1i8Mls5l6B6OZeXP1UAQwWqFfzUU6kU9arfPfffj4iY3Y2HjNl0CR2LCY8TqMY0FXn7ifuVxfpvLqW+758upSRGESGleti8u3n95+RkDrFoN/T4Dgy2kshsyMeBImFgudNBbCYg4HGlssFbdpFvOOzfo9Rk2AM0sRwN8RW4bAspB9zuJpSOAMujQA2rHFDl8eWfAV/rPYoHf0smPJRpc88lxqNYBmg+PnFjvabnUOw8A2Rz0F6uV2m4/81ltSq2N4c9wfQIfHstlR7+Q5NNu03v7b848X37/9NHnz4RJwNg5QfkDgDt5nwV3Gg8M33cEBTxLg74G45cvIFxMAOheJ0Wq1XDFjk5nni8R0O0PqNYxdERMr5PgnPID2bmjP/SkSWz2Nvbk9m94WT+ypLwK31ELS1vuaf0vg83+6GagJTz0ntMPpl9LX+eNl6pceg+zEIvHC4NC1E3xTAxp54itPNK6RAI7qGx47nu9zfQsDmHjBLLRRV0B28PMZsn0Mn0Q8XdhfQi8wXYvNOmwWxmzGvEBRx5sx3UjcekmamNvfdHKQzwDmrACRgPAL10R1tfFn61vjqdHpdMoDg/5maDvwcwAIakx3M9AoN7nx0oVpGrYfzg1Ldwv6nMXYWDM5yZZLHt+ZOGTFaQ6DJQqAJpoG/5IlKUiFxdYb2bkPqkkEyVGZGU+frvNP2qREbYu1/6bdGVobqVRPn7I//S+tWGzNZVO4b3c2LLrFl0YJoEh+zoQv0pCVAANvsLVZerTIkJfQWQ9eyBuLEX8ZWJMKyNJXkfqo3QV4Hey79DIR8wwMYjJJw5T7ahwblpSEamZ8F8ZLPmTr9jLzU28FzOZt5EcBpvSiw4SfCNbuA7fTsL2poGU2dT1D8NQrYfdZke3D+bA6pHbi1bpVFJ54RbcBdtnYjWpcdFQGfglSJOJhhQH0SBOuQpBf/vg/2GdPgA1PhozQZ7sHVtC0MhhmkiXt1IeED/PRwFjKgrRnVHkv9ZF969M4mAm6v/BWICvP2MpzBVyIAOx9zH0wKp1G4FM/p0Ez8KdPSW4eIlFPnyoSSuUsD9og8+qARQPdU4qZK5/NowiwMI3zohEIBowCLZg0GAhpXIFypYxHpRs/dCekn7s7+f6HN/BkLuJwq5OKlQUOXqOcrXKd7xhk3q4ttkITJVEqeryyvVQsE7PThNiSg9+dxML3xErsRu5SNhgybI9BUIACBo6LORibYHiVwFOfa4kx8r6MlYjBlYhkknjBBE0CdIfMB04ZGt8d/c6MCzQhQ/T07Bsme4B+4Tfx5plgHAM9bcKMqjeq6/64vQeT9tWG6dcsf9pRw0Dy3mhcleCAzQnRXo+vOjux/+Uf/+n//O8/svXNRgFSjsH4KfgpUMylTzrgKC4/vKPYyTjPUgxgDI5/QQjeZ0se0JMFXYEKUYAWe5xIDS+cjLtxFgmXPvgEmOZkwdcBBI8ZgNgof7TyxI2IJ4t06Zvu6HcgtxZbCG++SEfPj3pqPBDcvpVBNDt8w0yMQgW46w4qL8TD3izmSwHRaww6XRIAELpIpJwCbggWwDxjCIKRshII9J0uuHK6uMeRGyqMtxFV8MsFpRUlgcivXG/FkvTOF6O2HMRwLf9uottT10sin98NZ764PeWgYEGX1GHoCMwdTreFxkBX7M3uuiqW0y0RQtf1YuGkoOlDJ/SzZXA659Gw34OOpty5nschWKDhk/6gf9h/2QR7SiFMF2P2LBn2j+BDABTGwycn/MTl/HQGnXZnEIb7d8PkDmKCZTfz2mcNoMrjpq8S76sYHj2Pbttn//e//vf/+OoAGpxhq7NP2RRzGE6Okd2xKPMTzppgTjVEhZV4KU6EaJ+9E4GIeYySgGnSh3MwUG+1iL06mJ6p3uhXCXsSOyDPM+NApigTiMwPMNodretsbpfZDDb7b9IRqGxqYqJj4w8Yr43RqvFdS2DsjNpr+N20NfI3npsugC29b063JUKyYNirsaeJNds8bYPJ8cMboHjm+wkYPxGcstu4m8B4PO53IWV0rkHb2jXg8rPiI6CURL/G2ao4ozHtktAOY0T/FIJISNm6aRgNj5DPrzgDtZzl44fkDKzTqA3+kwfX7RqLjSpvX05fTmez00J2+ocI83wae7H00EFKthaSZUhcQKd/+ft/eHXACy6r6DbM0igD7QVOooG0wCgIndBEIAI+X05dED5IpRqi+weF802mHQNx3RRGjUNo+pSdQVoH/oPiGrR1EhhYLUAN73MTX04QAJAPyIC+Q25WsjuYDEDuw+NUhv4GRTQTo0PoGHMvDg+f94w8X6DHkTkrgSh638aWGn1ForkYAthfvUjykNJEeGyOv0ZXDTT7GqnIDfwRfJmnlRWlqdj9joVYQBivx6Bwt5fRkQEvCYJVy10Ub1uK9VEczsHIJxIkCLRjsWUyh9QesmQXaDv6jgNSFhNxHMbkabRcOCkm+fzWhJZLGH4fE2tUetI3gsWeouPv6MAFHMqIlUwC60I38k2K2ZQJDQ6Y+tDsw2u87pCE0dMz1rN7h3kCh/gVUkGAQDMwjX8ijo9eHrkGfkmYy2am8aQ/O3GfT+lF8bnx5Hh6PJ31lN1Lb3Fo6P7/y39ga/p+UwPVqqj8L//579j3wMUQtW4t/KHdm2HUXuml+skaqEwRern5M+wUH85wmiZgf7sGwpSBIZ28hAb/uwJ7o9Np7XapZRs46A0Gg6m2oH3IJ5PQB0l+MuAD9/D5ac2EDsCE1s1bxF2cyiKfycjINnq805r/2WclKy6+7rzBMDuiOxXpDZpqZT+nYZqGyyG5yi3I8EVwtgYmbl4d0HXhFAtTOejlnnsNfzbtszVI9OabwhXuw1g5pe244UX/Zd+tkRGwPA0hNp2BAxkuPNcVwX56SO8n8TnNuwJfWOqKkD4FbxUkHoUz9BGzj5Itjq3b7VwQUXwLfSjJRpdmtYYYzPK4Sw4fqG+e9Fwxt5R6WE9Opi/A4+j7TgkhTdXeN4xQpYyKEOPRcsX69nHCJHB0MV4AMdxpG6heCTuqzu4VkePsX1+LO/K0CUNQa5wMXZc6hkxaUqDHeps0bHzVJcR6G5QIAlrzgHOKjlIBVgskxGIIw2K+mME1OW4wmwsh9J8J8RdeiSUwdjIFewliHxUTpFVvNwMJlrOl0vpaNJU8kZfzzHN54AjqSEfxcAMpcCosmW/X4ElrHyPMhKYkEThofsoh69azsghQJn2TMMivlfEmgwojRWNChgRNKg1NPykF6twDUZnH9lsUHNOgWBTyNjnP66uYFHKHWYxxNoQIFKkuwi+UaNLUTtKxi4QyH3zRB96ByZUz7Db+QUeC87lPnx6iI+grDxLMoVkx9W12GoczLGj1BC1cWMMm7zfCMAAgAjQbbJCXTogI5jaziwBG9jSS38pckh7poDmlHHAGaRs4K5x6xQTuQqakgczvgHXwUKZtBuijutlI79OreskWPdXOGiQU3G0pGEnSMXR2pXyxugPn74c8hcalyAvfLWXLAuZNGF+bJXBpfFcNdPArxPdKkSnOgr9YS5rz+0pnZS3SikOZhLjtaO3JVeZeeJiSE4tIt+Cns0vDSkp1P5Z7lO7ej5UijiSDtFqSXuTqKq0zis+94GjSb4QUItp12KsRGGArD+xGeFHIgLh1RJSyt/QHLCMuO4kqw/UaFQRcAjlceZk/taMYOp0APLOzJS8o0igvSRqbQgkwOKdRsRJmf6YrU+U7KIUQb3KQlGD0Oc5UiA/RMUXqqotrITCwNsEWZZGL5rpjdSBUPKK3NwuId/ETL5lAyrUSZbG+84TvSgDPqgBY5aYaD+fapFQHw+KSQunAv2cf53YoH/6v1rdhyGhcRuA5/BIeMu6TISBRXWlsq+g9T/HisQEpEcKFK8wI8JLEB9MCzDzMKh79AgPKB4g7lnacU546C1NlGqRkSp1y+YMhXqhrs1O4H1qAwc/ucTQrHnsgo+hjEmOvE4BMC3DwQ3w5vsJ5vTwv9CyZGoogWyo/j12X5EOja3qQgfgiUA0snBZzRriOQUssQJ7+5mBdNNDzgtKZkIzPaII28QIQXTBS5gzFJ1bJ3cwOIJLZbW3jkpVFeETUUZMZIzqP6LeqgEgHPYMJPMYE9KraAmhUmuKE3CWfz5nSyjl4HbzrbNgvf/+fGl7G4zZAbV9VRv8Ay1LtF3OrHf0OISuqzbhK7lbmXdGuSSlc3k1ogTMxi6nPd9wXIDro9OXiZwJ+g8cuTgtjIg3cAa6LFTQJayFMp5jtxJlGkqdcltz7FyHlqr39w+8/47KyWpFEPwhJQKI0qDIhUZ5I9RI5ZVElHWZDXpAVkuMtUc4D8DqmuW/FFecGYKgTOVQ7CuYYgMhn5MbpUeP0zMMmdmBgxZyMXp5Yzms+BemoeW/Ca4ttsd7tjIf94ysbIiUfnIxpTBBTli/mKkEgUOPh896VYn4IQCX7TbFKh2hAPglfOOkbnnJFRiApsdH9NVnY2kErzUEKqnlwZ95PxJxf9/p7/e+BfFXr2RhcIw3GQCHbC1xxe9VqkL1HT+LrkpdFlnp+EWjSre2E0V0Vlr4BVIjlE5rlmkw6W32A4VUuFVwUBtQqNi7bS/UKCxBsCKZcEyVhG/lqpUI+F1UzVyVXwJOkLHDbs6M7Vv3lI/SjcN8fHB51tH1a4NxQfKet01eUxcfKH80kNsugNpGgFse5WoA3C7MY4mfda5V2knFRUmGcDv3C2FnkD1dLTHaoob3y4jTj/mQJgRqOJ2+Exog+s53M5Ta8n6CDRkqZFNyWXmJ8tsLqIwicTeUXMdfrlZxJk4OvTnGtFYnQQcfeVJjCjjob9A+X5x/BgayWdpZAVnlAOWRvaPdnG7jZXjMBUNBWLkbnjXHK7d23uEq7tCMR4wrS5hta+n938XuWJdzlw2ZY5nzcv4KMdT7uXXXqva/pZb2Xysp4ASpffKPldoc7C0hsKfYQNt3l/rfR995urbVROH6LQjl5/+HTZ8pTU8g9MVoy5lGmL1cxX+rr0qVTtAAbHNH1Ji+QoZxIy9q2eKGArFBIMIKPbuAH0pttuUSFLgroKAwyAghyPN5Nlh7rdn/ORHzXBWRHCNr7SnNMNtxbUiaJ6fqamGoZ+4wqDQVjQkjeCEwU3ojYdmN+A71RGUY6cpKVFYQLSFggnw3CLPDSxNibm0EC7PukolgsFyF0ZUfUQ8wX9SWk+fBudNiBbMeFqxwwToqPZaJ4K33ELVVRQbvYi8yOnLMwISoyOiBu+tYylNknwkOoCH1ElqQ6h3ZUeAd5E5fCiPMGfDzAx2Be+PjwCn+PChAoDxM5KQ4Crr7q9wZHJNYHa/w4fyJl2rjHyJZAGhAqo1bJT+61NivuJ+TNnNREMcDxAbDRCmfV4E+ur0B4eK1AweVEPcdUC5k+SqOiOOB6YknaInhdb1GgS/oyvp5c6RjmtjBJuE6evx8Puye9Sg0ZArTy8cLVEjmhA9cQRClUCzql2DXmMyxoBbVfeauQmSc9lnRsdhl+CZmT4TQS4/4cC2ATuVDve0tAGtRnwe9wEYTnIWxFvZqRAXLm+tt6QCyvq+Mqk9aVVfD22Uc5MggvGE7vwx1Ye4rry3O9O1K5L+E0kc95ijYrACYb+BAdrcGdFIuRcIIN44ZgguPFO1V49/2Hj2UBOXmhJORkIEXkpCdF4+RYCcLLQzV1BwmDLNkzDRRIZRWtYn0ILr/B4MQ0fgQvY2hbiS3k1Lhu0apHa6q5bq2WoQp4r2V/jupP9HjvsLcT2me0v9oOY/uj2fTQPcHLP/3La63+f7DYexjNc1yYO5HGFiMMkABnAelUNU/GdCmtpjoYo1mkGkBhi2UUNhONClnAjDeX/sKMr6gIdkEh8GLcBYtxNkLGVFo59aU6+oziAeiwnFDgi2pGQehqbUREKwkIOskFLhz3q19FNNtr6LossGZ/ADuIOf/LEzJm1vo9+O738BCXNG/JUnbItkkfvteXqKmG2+pUAwz/pDe8qi2BEyOKbPhVFPp3uD7CIkAsTUbtNeC6aeMciT9qB5BhYXFEHF6Dqq2xUmAPJjNDtuzSmhC0H9jHVfL27eeb9sFZKe8jiSjhgwtnWrX1kt9hdMtO8gKMyiphaWVqZxXXuq14fXjYrqLTVuuP7U2+JLfZXqTcDdjZBa59tkbZ2LBX07M1ySE5qHW2ket8tDyoqIBWBNMJMDMykteWJY9MoRNqdMZ6hVhVBJHCUp9vKGDDbxXo35K424QlJHcQVjfdaqgWajdNpUrQx+eYT/kXCEWRkjmN2soWy8JipCkGswQW2xEttikNzmpCso7xxCu6uu2P2r02ux2AtP4BpPsO7teohj27j7eDyq1WBDWW42P9qOvyZMHjmN+N2kfsCEWcesTQDWeOG+qjJDXOfvnHf2KfqBzPDevlm8RinBo1cCwG5TPSXP5UZCy0El/hrOpBmnro4R/+yD6H4Lll6aUmyV+5cl9lqMSqzvr6gnCDiGkpfImL+gO1qE/1ROy3XtKnsrybmEdD/KF6PFrXXyPTNo2r+ztiDvTJkPip6IaqK+STeioIJuBga5V8ZxKIRRrRjbz9gy4paC4RSFZzqs38NrwFeWY9BsLM1u9BZHHfhYhX4jyJhJNeYtpSmPUdNW8veg31F7ICAknEekAkrUubdbstpZU8CyojIPNwHimy17gBHWiwZMM6m1qZYKv1+hMWvWK0OQ3dO4vZsliQmM0BsZittysk2L+SGSKHtLrV8AWkJF1Fj+MeFpoU7U9ZRQw3rScLN2aLPluXijtQgpmiVE81idi6WqmpGwyQlAybJZ4rauhO++7g6JTtrJhhNWVCGrJyjQyCTfn8N4GL6NpTP8RlWNWLvK12Jm1HhYYKuiKI7LPClCdzLGUqQbmnMuSFc+TOZp0mRs1ms/Ljlg6W8qrFFzUGb9un2uvbLlh5N7zBkg82OIL3Co/jY8A89uYJpEQ2CjBbK8HeMP3c51P4XVc7GVSFbNOahWGKkqiVBFW10gAlXq2BZZ7vTjJPJ3A7kpoLtbQTqaVNsaRZrtimy8T+lifCjGIPa/Umi0yMDC9wvTkWDAQiS2NIhOlp4kNUuX/2AWk7Gheg34Xh3BffwVNT7rjEnMPIDTt6uoQHSReD+5lx1bET8O15B6jXk0ISJhSSGkqT4dum9xOXx9elRgUwFM8GaFJGEVpTAw1Ot6qDI15OSN6wFYlzAav0NgekmlQHiZMyORRpX/T4ilcahH5fw4XEqwpJ2pscn9RLfdHcV2luO8rSfURvbLCb6lmahsEkEWBjXZSvBgY4g9khIVlvux/T+0DnFH8A/J2kdRbCuQa9V9RtwF6zfH/LneyvfdYsCDsb1fCW+j4VVAqkdr3KEgUvXaDGf4tykJgkCCPj/IJ9xOUrrM3HdWZcbUe9HdEvdJskI/CyjRr/JRkZZmd0tnZDBxLOILVRWm3H50mCFak2eAyzjejhRrQOTuS4YlnarqNRugxvTOGL5cRzRwa4S6O2AAlNPgIUMLqYNj9hdaR/CrYm5C8q68OVjn7GaT21YYUKjGu9PWHdR/7DYnfAxvv6c+aJ2A0r4HTfr2nLh5k4ECePDqlseSJz5MMBVljo8aNjrROgDOczn1amB3e0MQ2168LY0bZcTgaffMASTJOEayQrDvIlaZSJuwiEJfJ8w3rwKiGjqk5aHR0Zl+++PTfyfUKDw15nJxw9iHPHAePphYFpfPzzPyeqfI6ZAoJg3+W4hTRWxR9YYdcB8LgI18jTXX2gQNzTlNQJNLmBTG8lJlX67Bl2f7Bn2Hmhgpg1seSCBosyZv+6/VHx3O4OwXv9ih3KMs/tzt5XSiRr/T0K+ERvgocu3sRhREbjXiKMZRmlRfWEVFpIZMHgRL6QJYb0Bts8RA3Ul/hNASfvAB/RTR2+cXU/7BX3MRrbhbMi6g9AiUDte0QNMR6lc/kGLWZeevNcu2oFJDskStL/ElMrE3fCmFhFhIVDEvFzuTkxWYQ30p+pnR4Pty2sMJgytH7c4M592pkcC+4/YmBSEOXYPkHCJGKzZw8sdmyxvv1C1mGPenavn7NgKdI4rx17qLHDXbXINr2x9hF2rVTGSVi+VlGDKVf7FFZqQy4Nn2qRdmzAvaeXGiFg1AMsOu7Zz3NSFJTA0D8A38bdx1HjNaCIHz2CCDTdkW6JoDphJJdCjJFy9C6AFlRPYTxKCFUlyEgGWnpTvcClCrmdnJnH6nCSZ+w7CIwWP7752LGZ8chOWOlQkSE7VgeeQBBBkoIb7YS88cFhh/ewD5hcs49jA9dP0SThGSb4Fw8pwepMSSpcvFfkGyPZrsaHV53HEkpRWp+TYEa3nXsQfaSXzmu998n+v8m8lMeYpbr30UnRCtfP90HUA5JbWx9ph+Te/z//z4DdyarKxwg6fVBjJR3Hkxva4kgehWypQwygZZf3iIsuNN9HhB8JFdwEyHAH426I01SZpm8pFcvD1ObNwUYpMJ6LAH2G9xXG5c/3dEHivb/EqXz0DhU4yYK1znj48monXNwDd7ufI1hALE/qSUzccUOZcjIak6e+yqn19ouANjsdQyWM/+glMtsJk32h/HyyRdlzBysYvK/Ef0m3ZLmHbnPuSwDv0J7Edzo4+xw6nGVBSKduMI47fH3abViclPUYqyn3viejQR7ZHWMCRHucZUmhuFHivxPoAgtaJLLfQQpABdGQDmdBCj4XD3UBDhRC/+8+XJQKcB9C9PMVD/D0o30Uf4R1ou0RFXd5SKu7Fqta1d6V8pz9v9C4XpCrMX/59/+tjzt55FEt9+i33PBSwa6P5QrPj7bw62v8DgdFlEmf7+mDNtJUweuDsOTfah8D3Qcel/UYMih8XuO5JWz5539GX/mQ0Osh/JNbxKDx77LlFEcwODzKKfBJLPF0KgM3HgjHwyOuRvekKvk2tJpNVRuetbH2BU/D2NsXj+n9SFUC42ljEJdiOKYiseMc6nfQhGIqxvUOuj3w5U6lCvRu/yXApR8NH8taFPh3HlrzxGNLD61FGmPubv7pXzq/DivyrVL1EOYI8XmBlqTfI/yOjnp5AHMxNvIPH5Jg7ZGvS5GAAZMu9CHuU23YqlDwSB0uB2ip14aW+kKsVOTL5IFOj4srvg9xM2R5/8qjpkemE31iwD0Glh5jrg4+zqB5AaPwct+V9s3s7unB2WI+6aJjg/sg/1Xh0b3Qt9ztBdWNguz7QPx7P5d7hcozm/d98ViPt0+pt7H/UcQsS2jWD9etD5gsQntA2BAXmO0Yy18woyoSRwBnvAfMpb4oT57iOmST017qjPj954/fm9XSx+2h4Y6sUvPqpjQs/HubYN0ulkLO1KR56bCZbXgY0pTglU/SaGiNxydmSY2iRgMHGhSf4vDkIbr+QJu7okMkJTIE3NxWlWJe9+Uev7eaQGbQAOidThr2gwEh27JKCsIbkBYez3FyGASf0ogUi12esfzwuARshrSCD7L9jdqlNpPq3cs8ivy7iZxskFWAtc3LwASwkCAbgLGOcbBdNcPQ9fG6NebaaakrGDdVS6LQWIodliSmVRDFUlIjtylftapzIbaz4MFcmGWULab/jpt3PkPqW9q7BwbDdnzPuTb1eQVNlBz/JmcY/DYnGfzV5xk0xhHErmdsjDbjCotXAyeLwYU7dxOqk8ZQBHcci5h8fvl9nuS2mua2KTlS+1oNeZJHySSg6VGz3p/x0uzbvdpbME/AvbLps9QxAEscVHMPFgNvm4l6PkbZppIHeUDTkJmlnZBWse2oo3sZQ35pycytJFfw0E5oy5xZbKbTnyAxSyJY6pT2/gLMqRZ/CiQs7fWv6BV8bklXu4PSpb09Jdj57iWNBnm4ctH9MpSbvv7yoRMMIq0JtoSDiZncJypLfjshd/z8qLL1CkHhkUqP97IVj+tjxhu3Jp/eXv749vKT3PemjxkGnZxg9YmJP8VegdeeiCHI90OUEpwL4MwRvssJWMh4kIK5vmPTLKEZBBZlIk5D5ntTdf51eR9f6FyLtNgUITeQegHW/uCM5I2Ylh07Zk4aVzsKI/O6YS+oWz6waFg7TOBu2+m5IAFhImrhy+6dJFvb9tI6VkaaBYHARYYCOdyR1oxYaqcixpA01TggKSIkRUwGHCllMaLXM5Yftpi7c0lDW/4xacU9ecCwE3vq4d5Yo2fT/zCTbYhLlLxFTcT54RNtnN+GXdk8rCEA/kq0fJ4FzmIi2UzDG714+VwPjAY62pK/XIPgXVEAVeiVBGpSzWM8QZc7Kg1NPaa+JD1l72pHFNbwil3rUTg9JYJ0Is+SmGApRf4ZmE95TIJ8oFct+02zGDTZJdwJ1UuOxvUdl+XpSWUwtFSNtUrgGRek+tumgHTWyU+ad/PisFtROWe81MTYsyUXPiuf+VLdIQcmai5oobT7M/3+gHO2t3vW8oxFmkbJ8OBgDiKbTW0nXJYwKSN1AI5W8EQkB1h2lqQHGAmj4T3YvX5ilL7vAqGy2y5fus+PyNniNEtt/zSM1VmA7cZRWqwXvjg+rlhXeFwVVbBDeCJ9ApYZvAgMNgOBGhkc98Jo+e1Xz5rQzav7fPEJTlTLE/ztNLwG95fFfsJnwjzp/P9UgLLMM56li5GJo7JyPBsWe7DZZAliABlXYx3Rw3Xlr1IP6RgpYJfJ6qDXW06Nutag+6ioDFC6fIRjxTBKRamqUPm4Dqe6U/SCNn+PSYIKg290u0HYxXN45TEr8hF0+5AcZEZaAkrSH7wgxvWHa+TV5iGzV3In56iM4YeLtxhjQ9Qdl59/+vwGqFraEVqKGWsnQeWORh51Uz1Jkb1iz3u1AzfkJgvsSW0ttVEO/OqeasrN8fQtAVEgZmtmnNuHMe9+7XVPulfPfrKBPQU3fkKbgZMdnjxVrp4bg9ottz2RZPjSxpK8yGzIMCEo4delA3aajwvSMd+Yzi5D/zxB/1wa51X1NKHKMUJVoVSycqXo9JD9j3T0kWkU/x2R2i5HtL55mIFD3jpTTUH4fULzF7iGE+CZF0z+l0xoBkj+t1Dkf+cklvVWoW3bRi0GrgdLzRZpyyqRsZGWoNnW7DYdjaL/eGdaiAMhRhih6W1g0ZbdUH4BWkvTXyDe+n/uxH5o", "estudio.html": "eNq1W1tv3Eh2ftevoNueadJms29qWeoW5diyfAnktWF5dhYQBLuarO6mRbK4JLvVUk8D+wvyEiDJWzAPCZCHAAHysG95WP+T+QX5CflOFW99kezZyRqwm6w6dc6pcz9V9OE9VzjpdcS1SRr4RzuH9KP5LBzbNZ7Ujg4nnLlHhwFPmeZMWJzw1K5N01FjvwZgORyygNu1mcevIhGnNc0RYcpDgF15bjqxXT7zHN6QL6YXeqnH/EbiMJ/bbcKReqnPj06SdOp6Qus+P2yqkZ3DJL2m334sRLpoNIbj/v12p91t7w8ajYiF3Mf7sO12dvHueyHv3++wjtvdw2s6T/v3+T4/4BxvwRRvB+zAZQxvzHH693vD3nDUwpu4BJbRgbs3XO48XAzFvJF4N1447g9F7PK4gZElycQcCvd6EbB47IX91mDCvfEk7bdbre8GQ+ZcjmMxDd3+jMU6cWoMHOGLOHsHN8ZgBLH027vRXEuuk5QHjalnNlgU+byhBswzPhZc++G1mbAwaSQ89kYDMePxyBdX/Ynnujxc7txPRbSIRAI5irDPhonwpykfYBTMRPOBz0epeoozDvHoeknks+v+yOfzwZhF/X0MMt8bhw0PlJO+A4XxeHDT8EKXz/u9AUE2rmKA0j/LHSvh48UKmo1dS5UYAyW3fps2KnzP1dQsKSifbMTM9aaJ4i1irkvy7kaKtU40V+S04TRNRWhawzRcVKilMcQTsRgsr0gZWi6ot9YoPa4QwrPWBhWlES+cQNDANI0ToIqEJyVxNYFgGiDj8H4oShlkTFlihaX7Hacz6rINpa+s6UN8bOhzdyGA1kuv+1a3l9N1+YhN/ZQWRAuSb7+NZ9ctVR1zn6XejMvho3WhfLsKtvDoupoV8HBaKDgUIR9sGpmyqJYytvY2y/+NNrCH58ALVbDotw9oLmDzRuZte63ZpPQINk2FZN4SEQ/XdjD0hXOJWRrUmKl+lR5WYQYZLXLklM/ThnSLPrnRpg9LAJc7ImZSMkpOGfP7ZFfE8eoGaU8VOcklhZXeYYIF8/0JbXllC2poiwXmi5KA+f5iwzsgN5b2pRoRSfg8usWOxGWhpioOGiaOG1e5PlpAMxRgKdgSk9REX/qaDEs9kjF570jEgfJj2DT/g97AjFGJPiuBRoYi14u5I9GDoWkQboteeWAji1FaPdibXYFDFnpBstgIgqSZ1Tg3+DxNUm903chyWIY5R5Eb0G/2u0qoys2ks+YHXw1Ra0xZ2/hCqtuqSDmuBu6PRiOg8sKR2KJClU2QtKTP7xGPm0ZFFoGkyZWmcy3uDmjfueu2rR6o+HzMQ3dxW2SR6zOrebzm+7u9dd+velVWCDjOXx16Cu9d285KSCz2VuxFc73ZdsvaYqBZ9QC0WmuZI/AWeQTCoqKs2Agj3QovXihFm8W4+3DqbTL1QhRriDEr3G3harvNb+o533x3ueOwcMaS9WB72MwqtsPEib0o1aiutGteQHVhwKLa0SJ7SWr9RS2dxJzX+rVJmkZJv9l03ND6nLjc92axFfK0GUZBUwL9Xctq77WsVnM49XxXjVmBcKc+x4qaqVA1oUkRJs1fh5LPWYAaLGl+ToJmbbnEJiTz2EVT1b5U+OENitY8167BEVC2apoccHyWJHYNSb4mJ6kKTuQ0AJRjai5LWWOGhXxeyxeIsHb0AclkGrPDpoI7WodHiQNBHb3z+A1LCqitqFU8JmCeiORWjCB+XTt6Gjue77MqxsMmNrOxKexz64Trqs36RLRWUMlmkVBqR7/8y5+00+mN9ss//U/JTQUFJaltYvKp65CdQC1vCbbvhgDnMFNPxIDMnjQdpm/cISqswg/cyuFY9jR/vJWEG7MAmd5htaPnePzyMz3fCg1fCAH5jn40PUbfk7BYS5WaS7aUTO+UrMOCYLtg//df/+HftOMvPyOUsF8tXceutcxW7egF1c58+z4A096XUCeofn2X3Qp2IKFOmSs07+aP1h1iB3CjAo2oZt2Ktdsz2+gtu83dW0Fa5n6LzDj2hrdzN0LlXjs6ZlUH+7rgURJtl/vJnOIW1HmnzHMcSgEr9PKfPIxQ0Jb8jVkImSCacV/88qd/34CTWaLwxGKcEvbmqEqeZXyiMVkl3OLKReBKhfDXAxeNxyJdCVpPkXgbGGSbTiMNN4VfHcO64grABtIrFHPwE+F/+c+xCLcELAJKLrlPRvjHKfd5KrbDTMAefOIf/6wds2glmpZRLf9ZSUoqe2DDKiFpDzWWaB9evT850UaxCLS6zBL1QT6/eBsPvfQYKTKGoJZVoDzvONlkcwUU+aeC5eXphxen0DzKt20ofDmVNEuw1eXvhQhOwpkXixAmlm7Fwcv5pLm2QGHbAadJqj2wE/vIFc5UzkDM8fUZRO2kItYTY5BBvXh9enJmLxDF+ud1ZaUfyaLgyNbYH9bNfFDFOe8mn7gwZQLDqkhmsGwwS1U0TLkqG6XMhCGmUpMaXOYs/O718Ym92Ea8X395+kzLRjT9ig9N7aVwRWpqH2SR8DkxSg5RZFqj4RyrXjz7Ax1TaTy3Lk3/IURHbmo/hDFnvqm98VAXCCze0bLlFlr30MXiZ/TLkWzUiHG7CBR7iVchVAIjz1FqEZYYfgbk22d/D6hUphhRIfsRuo95gsqu61pJ6gP07MOplo1++e9Q6z7X9CCosOELt/2x18pYOH37XEOm5rHQ0GWtQHU+tnsVqGB6nUO2e99JHiqq69dVJaLBFLXJFMoDrlKH/fobFjHEdk2OYa6qTDCNclKjvo9haua5XHxElvNCFltBtAuA39OY0qVD+yrAxrDeLmq1Khje//JfEgJS/BhRrmVWFI4B8DpgiJjEiJzU1GR9Wdh9LNXHYzvkV8rlrR/58OXp+2xcX7CQDik9lqBHnXKTJM3jGUcJcIVW4dl0NEKHQVNLuEmOzkK1/c6bc/89nQ3ob1g6sbA/XZ1/ljNmxzAG1UVn6DV0Lwx5/GN2RorHV9JNqujFNI2mCCuoys/oWMpWrJ+9f/msHCwRIwZyaCQCwxnk0+OTsxeeH3jOh3KuwskEZnuFYYuH8pjKph0OdooIQXWwhVVYcDxBHa4XS10EGZ8TUBE2EnQRvCLhM3rXjYEct8rOrQIid6G35uqMt8AUBTEPKmDv3rw/efMSWGJGkSpngvpZAkf4Q2Fty1UWhUdFmdavBUPdMNEA7BqWclueE4RZVskhHkd09DDjxywAUR0VSqmtZkVbhK5tdlqtgncn9SWqlYygg4C5VXYAz4T/HE0JaU6qgIap4X2PtJvybWNnEYe+2lYnJzyOvapoX+L1FfcjGHe7Ze6iFJt3eLetHjqd3Y6TawZJRKfFxRbQbouqq7zhyUSviAfFLn/JRcDT+FrvtLB9w6yoPTMqqs+Zr1dPPpeGPI8SsUXlBLWu1txuSLd597rZyedQn0P0Co/aesmoBCk4nfDAqzD6Cq9JNIGQT0k5MKyR/IMtd/Fnd9e09qq7puWl+U7DCqrn+eET89dwdYBhGloOS9IVFjGm/MlC30sOTo4O8ezum/RPlS5gC7KxF9xNdn+fMZBtVxFgERDsILtozhSqmvq+GSDcZI9MokjsxRJZFn161ePoHW4wFPPK6DMx72LwFexp36QqTFmOwkZlJ6FiM+Yp2z7DxooNqAJGjpc1DE3vjKahZESTXaN+aSxQoalFp/Yia/r656VkzfOOuWt2L8xy19CY2bZaJsn4wsz7v2xRx3XNroVlPRNCpmXDkbuPZVbPtA5oqVpG1WjRBKq1rLUHwwXEeWMXhOXivdbjFtEERhir/KvWFz1hhdkuaJx3zZ7ZaNPa0ajXe/zYJPlZYLrdM0fMT7BWdoiVddaued7Csp5apcwT6/YkXbVoeX55MQDT0s4oRJIpveJz/fS8daHMj84DwwR+ZZ+ety/kUH4cJO3OsqzT886FYQVTP/Ui//rMYT6L9VdIRF6wjrV7oYarWHclCzS6grjRefjKpL+N7kMgIxdaWda7yOy0UpRi+PHFExWm+2RTmyCvKyj2JGVvpN+jdcZiC41H1t5yZ1la2BD2rBsLOIM1jdAPItelsTf/UcRIWqREOgicwzMv+UkQpdd6njuGG15AtAkP+pkZEgHXhX20ADPC8hKKhEb+cnZJicBVY8Iit+F0KqsYACIBMQdI4PwZpT0qI4jJwRDD0TVQDMthg7Ksf6243sVcUPJvLGU3xWEZIDPOAu9XcBdg30yE5DMNIUx9aCyXVK8gQ0KyYl6R8wiewHV2Y8M3uG+3O9KnlfQziTo2YRrz9FgeKlYSx+9lowERG2aSA8lKaBvI4JWt6ik21xPr2kSeVaqRRNzq5NwkgMS6MR62rYOmnElZSEnXGonZwzzBoII0kGUKJMxmN8UknYBwm/vVAQnK1hzAseaPJAyKfJ0ZCt4Ric6Nh67pWNflbDZy86iAWYdXW0JiR3SCNJT26KAeVEPOYtttoriQryN6e9hTL8rG3sXis0oXSqe6qicKA6x6CLUeCVyEckZqt8yZ3RrcbeZZWWGXxjSYPbLHFktBCy03T0rBOLCydJDStDwxfpL9qolmt3/HsmaXzK3QyjeYBmAf6HV5gVE3LFWOfXhzan96sJDSlWWmnqLME6eCPjs4A+lwrBtLDQ9ffg7HU18k2l/+rD1YzLYAzb78R4xwz5PDYXz0YAELA9AL5FdX72D6yz9rNHi9bfBmZTD49Ih8VydOLdlAJU8+KbqVoaXqrZJP/Xodzs6S69DRCsXxOXqaRB8ZC2ggU0pssyvmoVLjqTPRR+YC6pkIt19/dfL0eb10XrQQl0uHERA3FtmgzDHL5c46oWQirnQ6za7kaYeFbmLLA4FzmroYkP2MPJ/L2mCA7k7PikZNjBS4ASuStYI1YcS3sZDwo8EQffblMgvtNGYojnJ9UiUBfcobBSu7aLDrdIdRL81jnG1dlR0W/TylfegSYZY4YNjGQuUY9ARixuXIEjNlcbMGUJkYrFVAS5URYNtyxVfcRmyUhmJbSSvQqUxRAgXHoIBKXmplICM90JGsbduu0yFJ3ZCRWpbTa1X5GcKci7qmrLbVFQ4KoYOh4+yZ8ITxJORJ0rd6PZO+3fHVW6unfG6FWnZQU6dNfDNFiAHdlOzksua5QvTxErlEUSqrV9LFYD1tBKH0e7TP1rWUcBEkrht2EGa9QTmI8EVNy+oIkJZl8KAsgwdkXvJQdCVe1OuZvSA60SGEBLd8Ho5TKDLDVGz/aQ7yhiaKXeSGKa/u7PP6ew6W6KDkWJ110JOI0ffh4Yz5U5dlT6l8eAZHkQ/vRMI/fqhfKJwrHCUiTnWdmUPDPtIlHRVd38LRLPoWy3h0cGB8d3DQWJsdrsxm7JY+65DPVikZi0xm545ceWFLIViO70VP5QwlJ4kl3/bQLk4LHPh3yrPWVq+rU9k6FSXUbh9n34gpzHAJuLfD9fpH7F2TUCIEHXQrOrZJvq8rUGNFe9XTiGFWGkngnPNcARdP8qd+dYuooDOsamtZsqQ+XmKj+Ia4dYYQkPnkbe5erf/yFdkJETFMQQT85l8A2fcymOqkPGU/RXzPo1BdCoy8hac/ohfUCZjOzleA6diZeWGiwIt8qO4NNiLounf/9FM+Ik9r60/q8jK33q/ThXd9A1vpL+uYntQPh8X94xEly2PmMk3GIM3WpqFKbBbNPBewSWxLC/gNdhKams80+mSF0aArhj6z6n2pznX2iEh+Iyqp1B+pjKqOKn/6CWnzUV1TbybIssqBZV85uSpcjYFqRaHSU3XYlpVFg9vzTyGU7efmT30fS+UlsJZbPKrO+IQh6Q7to2FFb6kYj32lZHNo0bUV1GzNsFeZdmXBlvX0FGxUFCtLOGnmYV6YoXrNLD68GFBGZXk6VQmQMHz/ffZwz7aZkT2jknT52yl6xK4xYBaddKKdl6OvQxq0JCFjkLPBvrb76kcp37r7akiw7XAjHhhGtXjNvEGEsseT27ojBetP45hd41X+6mUyM56Uz/3z8vmiZDqwjwKLPE5ajA2KyF/L4qxFmo1dzw4v6pWz0k255B/trYskj3McfHOYm4hQyEdsLANUkRBde2ipTx6ziJqNC9vdHgoQF1XsuJulCjdz+2i+LQZJTIY0KmFUqVH2zqYpqReEMH4ywwMB0TGtXpc7rJsUyv+f2DHulPW5upG/uE3WxMii0F/pfP56RFh+Cx3nK3RWT29VGqEIE4v0zqjvUcrOWXMoBtKdNuqxX91hZ6CpvTmJBtZUhyFz1FavHlqtx9Sk5sXMer+bAjxFf52ijwVwr6eazGrDmq71nSoOLYua/ZyZKCUqO7MQWz2UCCakEbBI/900GFLhnR0vmPxrargz4FZ1IZuaSqwlMyo1cZfW7q2+b9VfNaStghtLRcdJ4zU6eSbaqaT2VT6Um4f2vW9I/oOtMFW+EMEGlfiZ8ZUVIHcQ3ihRthDegFknrGyawna1pyrvCrIzs+yiQpbUJWARni2XR+nkA4Ju5kvbYOj7OPXfDzrVo/ZKW6fs+2+ZORaBVflUXTV6gZXdgNjyBkRmk+w071sb0MHfmumcw/ZgdQPqqEDynNvNZNNxiost9aDLbZkIJHkUYrf2CKxOJcgE2dbecj0Gc3oOx/3h/ale9+iOt0n3vbQCbXRI5Ro6ZPURiLwJxoRkSycr38xJdHt+w1VSWlCcY/Kaz95+tXf3Odu33uauXxcXTeSpEJFecTsXnRGV4RTXn3M0h3rWoXz//Uqj4m4G29ukT7R3dHnKIwnB7OgISd7nqEMUvXqMVI+98UdZV39OZF8hfyHL4gCpjOio24uboLdDEo41Q2/LE1WYs9iZeDOB0nyxVP3J2kHRr/uuxKx+lFD9AmH1i4MLA2ypW8JtNDM+L/l1otP3JcYauJIFdB14CYIH0sy5ZdHvhcxR2WmZcj4FWxzNGeq8S+HKdvzXZq+yYVTHbpXkdWElArkDPKycr1UOTwObnDT7HK3+bVLYOK77FU47qvoivZTNIiE/H108+qQdyv+acPRgMcrzvoVNR7B/g769lZOfEHmqrT2TUXKNd8km8XsvQ/7beK22IKOt9JWaZQ1RBzQJdGlQ7i4+GUZLSt8KHzbV/6X7PwSGMIY="}""")
for _n, _d in _FILES.items():
    open(f"{APP}/ap3d/{_n}", "wb").write(zlib.decompress(base64.b64decode(_d)))
if APP not in sys.path:
    sys.path.insert(0, APP)
print(subprocess.run("nvidia-smi --query-gpu=name,memory.total --format=csv,noheader",
                     shell=True, capture_output=True, text=True).stdout.strip() or "SIN GPU: cambia el entorno a GPU")
import psutil
print(f"RAM del sistema: {psutil.virtual_memory().total/2**30:.0f} GB · CPU: {os.cpu_count()}")
print("Proyecto listo en", APP)

## 2. Instalar dependencias y Blender 4.2 LTS

In [ ]:
# @title 2. Instalar dependencias, Hunyuan3D-2 y Blender 4.2 LTS
USAR_CACHE_EN_DRIVE = False  # @param {type:"boolean"}
# @markdown Si lo activas, las piezas compiladas se guardan en Drive y la próxima sesión se instalan en segundos.
import os, subprocess, sys, glob, shutil, time
t0 = time.time()

def sh(cmd, check=True):
    r = subprocess.run(cmd, shell=True, capture_output=True, text=True)
    if r.returncode != 0:
        print(r.stdout[-3000:], r.stderr[-3000:])
        if check:
            raise RuntimeError("Fallo: " + cmd)
    return r.stdout

# --- sistema (Blender sin ventana necesita estas librerias)
sh("apt-get -qq update && apt-get -qq install -y libxi6 libxkbcommon0 libsm6 libxrender1 "
   "libxxf86vm1 libxfixes3 libgl1 libegl1 libglu1-mesa libopengl0 > /dev/null", check=False)

# --- Python
sh(f"{sys.executable} -m pip install -q 'gradio>=5.9,<6' 'diffusers==0.32.2' 'transformers==4.49.0' 'accelerate>=1.2,<1.6' "
   "einops omegaconf trimesh pymeshlab pygltflib xatlas rembg onnxruntime-gpu "
   "opencv-python-headless ninja pybind11 psutil hf_transfer 'huggingface_hub>=0.25'")
print(f"Paquetes listos ({time.time()-t0:.0f}s)")

# --- Hunyuan3D-2
if not os.path.exists("/content/Hunyuan3D-2"):
    sh("git clone -q --depth 1 https://github.com/Tencent/Hunyuan3D-2 /content/Hunyuan3D-2")
sh(f"cd /content/Hunyuan3D-2 && {sys.executable} -m pip install -q -e . --no-deps")

import torch
cap = "%d.%d" % torch.cuda.get_device_capability(0) if torch.cuda.is_available() else "8.0"
os.environ["TORCH_CUDA_ARCH_LIST"] = cap            # compila solo para tu GPU: mas rapido
os.environ["MAX_JOBS"] = str(os.cpu_count())
tag = f"py{sys.version_info.major}{sys.version_info.minor}_torch{torch.__version__.split('+')[0]}_sm{cap}"
cache = None
if USAR_CACHE_EN_DRIVE:
    from google.colab import drive
    drive.mount("/content/drive")
    cache = f"/content/drive/MyDrive/ap3d_cache/{tag}"
    os.makedirs(cache, exist_ok=True)

def have(mod):
    return subprocess.run([sys.executable, "-c", f"import {mod}"], capture_output=True).returncode == 0

if not have("custom_rasterizer"):
    whl = glob.glob(f"{cache}/custom_rasterizer*.whl") if cache else []
    if whl:
        sh(f"{sys.executable} -m pip install -q {whl[0]}")
        print("custom_rasterizer desde la cache de Drive")
    else:
        print("Compilando custom_rasterizer (3-5 min)...")
        d = "/content/Hunyuan3D-2/hy3dgen/texgen/custom_rasterizer"
        sh(f"cd {d} && {sys.executable} -m pip wheel -q . --no-build-isolation --no-deps -w /content/whl")
        w = glob.glob("/content/whl/custom_rasterizer*.whl")[0]
        sh(f"{sys.executable} -m pip install -q {w}")
        if cache:
            shutil.copy(w, cache)

d = "/content/Hunyuan3D-2/hy3dgen/texgen/differentiable_renderer"
if not glob.glob(f"{d}/mesh_processor*.so"):
    so = glob.glob(f"{cache}/mesh_processor*.so") if cache else []
    if so:
        shutil.copy(so[0], d)
    else:
        sh(f"cd {d} && c++ -O3 -Wall -shared -std=c++11 -fPIC "
           f"$({sys.executable} -m pybind11 --includes) mesh_processor.cpp "
           f"-o mesh_processor$({sys.executable}-config --extension-suffix 2>/dev/null || "
           f"python3-config --extension-suffix)", check=False)
        if not glob.glob(f"{d}/mesh_processor*.so"):
            sh(f"cd {d} && {sys.executable} setup.py build_ext --inplace", check=False)
        if cache:
            for f in glob.glob(f"{d}/mesh_processor*.so"):
                shutil.copy(f, cache)
print(f"Hunyuan3D-2 listo ({time.time()-t0:.0f}s)")

# --- Blender 4.2 LTS
if not os.path.exists("/content/blender/blender"):
    url = "https://download.blender.org/release/Blender4.2/blender-4.2.3-linux-x64.tar.xz"
    sh(f"wget -q {url} -O /content/blender.tar.xz && tar -xf /content/blender.tar.xz -C /content "
       "&& mv /content/blender-4.2.3-linux-x64 /content/blender && rm /content/blender.tar.xz")
print(sh("/content/blender/blender -b --version | head -1").strip())
print(f"Instalación lista en {time.time()-t0:.0f}s")

## 3. Comprobar GPU, Blender y modelos de IA (carga todo en RAM y GPU)

In [ ]:
# @title 3. Comprobar GPU, descargar modelos, precargarlos en la RAM y cargarlos en la GPU
CARGAR_MULTIVISTA = True  # @param {type:"boolean"}
CARGAR_TEXTURA = True  # @param {type:"boolean"}
MODO_CALIDAD_TAMBIEN = True  # @param {type:"boolean"}
# @markdown Turbo (5 pasos + FlashVDM) siempre se carga: la forma sale en segundos. Con la casilla de arriba también queda cargado el modelo de 50 pasos para el preajuste "Maxima".
import os, sys, time
os.environ["HF_HUB_ENABLE_HF_TRANSFER"] = "1"
os.environ.setdefault("PYTORCH_CUDA_ALLOC_CONF", "expandable_segments:True")
sys.path.insert(0, "/content/ap3d_app")
sys.path.insert(0, "/content/Hunyuan3D-2")
# Hunyuan3D-2 necesita transformers 4.49: con versiones nuevas su codificador
# de imagen (DINOv2) no carga ("Missing key(s) ... main_image_encoder").
import importlib.metadata as _md, subprocess
if _md.version("transformers") != "4.49.0" or not _md.version("diffusers").startswith("0.32"):
    subprocess.run([sys.executable, "-m", "pip", "install", "-q", "transformers==4.49.0",
                    "diffusers==0.32.2", "accelerate>=1.2,<1.6"], check=True)
    if "transformers" in sys.modules:
        raise SystemExit("Versiones corregidas. Ve a Entorno de ejecución › Reiniciar sesión "
                         "y vuelve a ejecutar la celda 1 y después esta celda 3.")
import torch
from huggingface_hub import snapshot_download
from ap3d import config
from ap3d.engine import get_engine

print("CUDA:", torch.cuda.is_available())
P = config.detect()
print(config.describe(P))

t = time.time()
pats = ["hunyuan3d-dit-v2-0-turbo/*"] + (["hunyuan3d-dit-v2-0/*"] if MODO_CALIDAD_TAMBIEN else [])
if CARGAR_TEXTURA:
    pats += ["hunyuan3d-paint-v2-0-turbo/*", "hunyuan3d-paint-v2-0/*", "hunyuan3d-delight-v2-0/*"]
snapshot_download("tencent/Hunyuan3D-2", allow_patterns=pats, max_workers=16)
if CARGAR_MULTIVISTA:
    snapshot_download("tencent/Hunyuan3D-2mv", allow_patterns=["hunyuan3d-dit-v2-mv-turbo/*"]
                      + (["hunyuan3d-dit-v2-mv/*"] if MODO_CALIDAD_TAMBIEN else []), max_workers=16)
print(f"Modelos descargados ({time.time()-t:.0f}s)")

eng = get_engine()
if P["preload_weights"]:
    eng.preload_weights()          # deja todos los pesos en la RAM
eng.load(want_mv=CARGAR_MULTIVISTA, want_paint=CARGAR_TEXTURA, want_turbo=True,
         want_quality=MODO_CALIDAD_TAMBIEN)
print("Todo listo. Sigue con la celda 4 o 5.")

## 4. Estudio

In [ ]:
# @title 4. Estudio (dentro del notebook)
import sys; sys.path.insert(0, "/content/ap3d_app")
from ap3d.app import launch_inline
launch_inline(port=7860)

## 5. Abrir la página web con contraseña

In [ ]:
# @title 5. Abrir la página web con contraseña (para el celular)
CONTRASENA = ""  # @param {type:"string"}
# @markdown Déjala vacía para generar una al azar. El usuario es **ap**. La página deja de funcionar al cerrar la sesión de Colab.
import sys; sys.path.insert(0, "/content/ap3d_app")
from ap3d.app import launch_web
url, user, pw = launch_web(CONTRASENA, user="ap", port=7861)
print("Página web:", url)
print("Usuario:", user, "· Contraseña:", pw)

## 6. Guardar en Drive

In [ ]:
# @title 6. Guardar todos los resultados en Google Drive
CARPETA = "AP_Model_3D"  # @param {type:"string"}
import os, shutil, glob
from google.colab import drive
if not os.path.exists("/content/drive/MyDrive"):
    drive.mount("/content/drive")
dst = f"/content/drive/MyDrive/{CARPETA}"
os.makedirs(dst, exist_ok=True)
n = 0
for z in glob.glob("/content/ap3d_salidas/*.zip"):
    if not os.path.exists(os.path.join(dst, os.path.basename(z))):
        shutil.copy(z, dst); n += 1
print(f"{n} modelos nuevos copiados a {dst}")

## 7. Memoria

In [ ]:
# @title 7. (Opcional) Liberar GPU / ver uso de memoria
LIBERAR_MODELOS = False  # @param {type:"boolean"}
import gc, torch, psutil, sys
sys.path.insert(0, "/content/ap3d_app")
from ap3d.engine import get_engine
e = get_engine()
if LIBERAR_MODELOS:
    e.shape = e.shape_mv = e.paint = None; e.ready = False
    gc.collect(); torch.cuda.empty_cache()
e.log_mem()